<a href="https://colab.research.google.com/github/syriascitech/Computer_Vision_Course/blob/main/exercises/Week5/Week_5_Deep_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl">
<h1>Week 5 — Deep Learning &amp; CNN Exercises</h1>
<p style="direction: rtl; text-align: right;">مرحباً بكم في مختبر الأسبوع الخامس التطبيقي! 👋</p>
<p style="direction: rtl; text-align: right;">في هذا الأسبوع انتقلنا من <strong>الانتشار الخلفي (Backpropagation)</strong> المحسوب باليد على دارة صغيرة، إلى تدريب شبكة عصبية كاملة على <strong>MNIST</strong>، ثم اكتشفنا <strong>مشكلة التسطيح (The Flattening Problem)</strong> وحللناها بالشبكات الالتفافية <strong>CNN</strong>.</p>
<p style="direction: rtl; text-align: right;">هذه التمارين العشرة تتبع <strong>نفس ترتيب الدرس ونفس أسلوبه البرمجي</strong> (PyTorch + NumPy + Matplotlib). كل تمرين يطلب منك كتابة كود Python حقيقي — لا أسئلة نظرية، بل تنفيذ فعلي ثم تحقق آلي من النتيجة.</p>
</div>

<div dir="rtl">
<h2>Learning Objectives</h2>

<p style="direction: rtl; text-align: right;">بعد إتمام التمارين العشرة ستكون قادراً على:</p>

<ol style="direction: rtl; text-align: right;">
<li>تفكيك أي معادلة إلى <strong>دارة من بوابات (Computational Graph)</strong> وتنفيذ <strong>التمرير الأمامي (Forward Pass)</strong> برمجياً.</li>
<li>تنفيذ <strong>التمرير الخلفي (Backward Pass)</strong> يدوياً باستخدام <strong>قاعدة السلسلة (Chain Rule)</strong> وتوزيع "اللوم" على المدخلات.</li>
<li>حساب التدرجات عددياً والتحقق منها بـ <code>autograd</code> في PyTorch، وفهم <strong>تجميع التدرجات عند التفرّع (Branching)</strong>.</li>
<li>تنفيذ <strong>دوال الخسارة (Loss Functions)</strong>: <code>MSE</code> و <code>Cross-Entropy</code> يدوياً ومقارنتها بدوال PyTorch.</li>
<li>تنفيذ <strong>الانحدار التدرّجي (Gradient Descent)</strong> بيدك وفهم أثر <strong>معدل التعلم (Learning Rate)</strong>.</li>
<li>تجهيز بيانات <strong>MNIST</strong> عبر <code>torchvision</code> و <code>DataLoader</code> وفحص أبعاد الدفعات (Batches).</li>
<li>بناء شبكة <strong>Fully Connected</strong> وكتابة <strong>حلقة التدريب (Training Loop)</strong> بخطواتها الأربع وتقييم الدقة.</li>
<li>إثبات <strong>مشكلة التسطيح</strong> عملياً عبر تجربة خلط البكسلات (Pixel Shuffling).</li>
<li>تنفيذ <strong>الالتفاف ثنائي البعد (2D Convolution)</strong> من الصفر وتطبيق <strong>Kernels متعددة</strong> لإنتاج <strong>Feature Maps</strong>.</li>
<li>تنفيذ <strong>التجميع الأعظمي (Max Pooling)</strong> وبناء شبكة <strong>CNN</strong> كاملة ومقارنتها بالشبكة العادية.</li>
</ol>

</div>

<div dir="rtl">
<h2>How to Use This Lab</h2>

<ol style="direction: rtl; text-align: right;">
<li><strong>اقرأ التمرين</strong> كاملاً قبل كتابة أي سطر كود.</li>
<li><strong>افهم الهدف</strong> من قسم 🎯 Learning Objective و 🧩 Concept.</li>
<li><strong>نفّذ المهمة</strong> المذكورة في 📝 Task خطوة بخطوة.</li>
<li><strong>اكتب الكود</strong> في قسم 🧑‍💻 Your Solution داخل خلية جديدة في النوتبوك.</li>
<li><strong>شغّل الـ Self-Check</strong> — إذا مرّت كل الـ assertions فأنت على الطريق الصحيح.</li>
<li><strong>لا تفتح الحل</strong> إلا بعد محاولة جادة. الحل موجود في قسم مطوي 💡 Show Solution.</li>
</ol>

<blockquote style="direction: rtl; text-align: right; border-right: 4px solid #ccc; padding-right: 10px; margin-right: 0;">
<p style="direction: rtl; text-align: right;">💡 التمارين مترابطة: المتغيرات التي تنشئها في تمرين قد تُستخدم في التمرين الذي يليه (مثل <code>train_loader</code> و <code>fc_model</code>). نفّذها بالترتيب في نفس النوتبوك.</p>
</blockquote>

</div>

<div dir="rtl">
<h2>Environment Setup</h2>

<p style="direction: rtl; text-align: right;">شغّل هذه الخلية أولاً — نفس مكتبات نوتبوك الأسبوع الخامس تماماً:</p>

</div>

In [ ]:
# ⚠️ شغّل هذه الخلية أولاً
!pip install arabic-reshaper python-bidi -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm
import arabic_reshaper
from bidi.algorithm import get_display

# دالة مساعدة: تجهّز أي نص عربي ليظهر صحيحاً في matplotlib
def ar(text):
    return get_display(arabic_reshaper.reshape(text))

torch.manual_seed(42)   # نفس البذرة المستخدمة في الدرس
print("✅ جاهز! | PyTorch:", torch.__version__)

<div dir="rtl">
<h2>Exercise 1 — التمرير الأمامي في دارة حسابية | Forward Pass in a Computational Graph</h2>

<h3 style="direction: rtl;">🎯 Learning Objective</h3>
<p style="direction: rtl; text-align: right;">أن تُفكّك معادلة رياضية إلى <strong>بوابات بسيطة (Gates)</strong> وتنفّذ <strong>التمرير الأمامي</strong> برمجياً مع الاحتفاظ بالقيم الوسيطة، ثم تقيس <strong>المشتقة كحساسية</strong> عددياً.</p>

<h3 style="direction: rtl;">🧩 Concept</h3>
<p style="direction: rtl; text-align: right;"><strong>الدارة الحسابية (Computational Graph)</strong> — التمرير الأمامي (Forward Pass) — <strong>المشتقة = الحساسية</strong>:</p>
<p style="direction: rtl; text-align: center; font-size: 1.1em;">$$\frac{df(x)}{dx} = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$</p>

<h3 style="direction: rtl;">📖 Scenario</h3>
<p style="direction: rtl; text-align: right;">قبل أن نطلب من PyTorch حساب أي تدرّج نيابةً عنّا، يجب أن نفهم ما الذي يحسبه فعلاً. الشبكة العصبية بأكملها ما هي إلا <strong>دارة عملاقة</strong> من بوابات الجمع والضرب والـ max. سنبدأ بأصغر دارة ممكنة — نفس دارة الدرس — ونخزّن كل قيمة وسيطة، لأن <strong>التمرير الخلفي لن يعمل بدونها</strong>.</p>

<div align="center">
<img src="media/09-backprop-exercise-circuit.png" width="600">
</div>

<h3 style="direction: rtl;">📝 Task</h3>
<p style="direction: rtl; text-align: right;">المعادلة المطلوبة هي نفسها التي رأيتها في الدرس:</p>
<p style="direction: rtl; text-align: center; font-size: 1.1em;">$$f = 2 \cdot \big( x \cdot y + \max(z, w) \big) \qquad \text{مع} \quad x=3,\ y=-4,\ z=2,\ w=-1$$</p>
<ol style="direction: rtl; text-align: right;">
<li>اكتب دالة <code>forward_circuit(x, y, z, w)</code> تُفكّك المعادلة إلى <strong>أربع بوابات</strong> بالترتيب:
   <ul style="direction: rtl; text-align: right; margin-top: 5px;">
   <li>بوابة ضرب: <code>p = x * y</code></li>
   <li>بوابة أعظم: <code>m = max(z, w)</code></li>
   <li>بوابة جمع: <code>s = p + m</code></li>
   <li>بوابة ضرب بثابت: <code>f = 2 * s</code></li>
   </ul>
</li>
<li>أعِد <strong>قاموساً (dict)</strong> يحتوي على القيم الوسيطة الأربع بالمفاتيح: <code>'p'</code>, <code>'m'</code>, <code>'s'</code>, <code>'f'</code>.</li>
<li>نفّذ الدالة على القيم المعطاة وخزّن النتيجة في متغير اسمه <code>circuit_values</code>.</li>
<li>اكتب دالة <code>sensitivity(x, y, z, w, h=1e-5)</code> تحسب <strong>المشتقة عددياً</strong> بالنسبة لـ <code>x</code> باستخدام تعريف النهاية:<br>
   $$\frac{f(x+h) - f(x)}{h}$$</li>
<li>خزّن الناتج في متغير اسمه <code>sensitivity_x</code> واطبعه.</li>
</ol>

<h3 style="direction: rtl;">Requirements</h3>
<ul style="direction: rtl; text-align: right;">
<li><strong>المكتبات:</strong> لا تحتاج أي مكتبة خارجية — Python خالص (يمكنك استخدام <code>max</code> المدمجة).</li>
<li><strong>الدوال المطلوبة:</strong> <code>forward_circuit(x, y, z, w)</code> و <code>sensitivity(x, y, z, w, h=1e-5)</code>.</li>
<li><strong>المتغيرات المطلوبة:</strong> <code>circuit_values</code> (dict) و <code>sensitivity_x</code> (رقم).</li>
<li><strong>المخرجات:</strong> <code>forward_circuit</code> تُعيد <code>dict</code> بأربعة مفاتيح بالضبط: <code>p</code>, <code>m</code>, <code>s</code>, <code>f</code>.</li>
<li><strong>قيود:</strong> ممنوع استخدام <code>torch.autograd</code> أو أي مكتبة اشتقاق تلقائي في هذا التمرين — نريد الحساب اليدوي.</li>
</ul>

<h3 style="direction: rtl;">Expected Result</h3>
<ul style="direction: rtl; text-align: right;">
<li>ستحصل على قيمة نهائية سالبة للـ <code>f</code>، وقيم وسيطة تُظهر أن بوابة الـ <code>max</code> اختارت <code>z</code> وليس <code>w</code>. أما <code>sensitivity_x</code> فستكون <strong>قريبة جداً من عدد صحيح سالب</strong> — وهذا الرقم بالذات سيظهر لك مرة أخرى في التمرين الثاني كتدرّج محسوب بقاعدة السلسلة. الإشارة السالبة تعني: <strong>لو زدنا x قليلاً، ستنقص f</strong>.</li>
</ul>

</div>

### 💡 Hints

<details>
<summary>💡 Hint 1</summary>

<div dir="rtl">

لا تحسب `f` في سطر واحد. الفكرة كلها في **التفكيك**: كل بوابة سطر مستقل، وكل ناتج بوابة يُخزَّن في متغير له اسم. هذه المتغيرات الوسيطة هي ما سنحتاجه لاحقاً في التمرير الخلفي.

</div>

</details>

<details>
<summary>💡 Hint 2</summary>

<div dir="rtl">

للمشتقة العددية: استدعِ `forward_circuit` **مرتين** — مرة بالقيم الأصلية ومرة بـ `x + h` — ثم اقسم فرق قيمتَي `f` على `h`. تذكّر أنك تحتاج المفتاح `'f'` من القاموس الناتج.

</div>

</details>

<details>
<summary>💡 Hint 3</summary>

<div dir="rtl">

القالب:

```python
def forward_circuit(x, y, z, w):
    p = x * y
    m = max(z, w)
    # ... أكمل s ثم f
    return {'p': p, 'm': m, 's': s, 'f': f}

def sensitivity(x, y, z, w, h=1e-5):
    f_base  = forward_circuit(x, y, z, w)['f']
    f_moved = forward_circuit(x + h, y, z, w)['f']
    return (f_moved - f_base) / h
```

</div>

</details>

### 🧑‍💻 Your Solution

In [ ]:
# TODO: أكمل التنفيذ

# Step 1: عرّف دالة التمرير الأمامي مع البوابات الأربع
def forward_circuit(x, y, z, w):
    # p = بوابة الضرب
    # m = بوابة الأعظم
    # s = بوابة الجمع
    # f = الضرب بـ 2
    pass

# Step 2: نفّذها على القيم x=3, y=-4, z=2, w=-1
circuit_values = None

# Step 3: عرّف دالة الحساسية العددية بالنسبة لـ x
def sensitivity(x, y, z, w, h=1e-5):
    pass

# Step 4: احسب sensitivity_x واطبع كل النتائج
sensitivity_x = None

# Write your solution below

### ✅ Self-Check

In [ ]:
assert 'forward_circuit' in dir(), "الدالة 'forward_circuit' لم تُعرَّف."
assert 'circuit_values' in dir() and circuit_values is not None, "المتغير 'circuit_values' لم يُنشأ."
assert isinstance(circuit_values, dict), "يجب أن تُعيد forward_circuit قاموساً (dict)."

for key in ['p', 'm', 's', 'f']:
    assert key in circuit_values, f"المفتاح '{key}' مفقود من القاموس الناتج."

assert abs(circuit_values['p'] - (-12)) < 1e-6, f"قيمة p خاطئة: {circuit_values['p']} (المتوقع -12)."
assert abs(circuit_values['m'] - 2)     < 1e-6, f"قيمة m خاطئة: {circuit_values['m']} — بوابة max تختار الأكبر."
assert abs(circuit_values['s'] - (-10)) < 1e-6, f"قيمة s خاطئة: {circuit_values['s']} (المتوقع -10)."
assert abs(circuit_values['f'] - (-20)) < 1e-6, f"قيمة f خاطئة: {circuit_values['f']} (المتوقع -20)."

assert 'sensitivity_x' in dir() and sensitivity_x is not None, "المتغير 'sensitivity_x' لم يُنشأ."
assert abs(sensitivity_x - (-8)) < 1e-2, f"الحساسية بالنسبة لـ x خاطئة: {sensitivity_x} (المتوقع ≈ -8)."

# التحقق من أن الدالة تعمل على قيم أخرى أيضاً (وليست أرقاماً مكتوبة يدوياً)
other = forward_circuit(1.0, 2.0, -5.0, 3.0)
assert abs(other['f'] - 10) < 1e-6, "الدالة لا تعمل بشكل عام — تأكد أنك تستخدم المعاملات وليس أرقاماً ثابتة."

print("✅ Basic checks passed.")
print(f"   f = {circuit_values['f']} | df/dx ≈ {sensitivity_x:.4f}")

### 💡 Solution

<details>
<summary>💡 Show Solution</summary>

#### Approach

<div dir="rtl">

نُفكّك المعادلة إلى بوابات بسيطة نعرف مشتقة كل واحدة منها، ونخزّن كل ناتج وسيط في متغير. ثم نقيس الحساسية عددياً بتحريك `x` بمقدار ضئيل جداً `h` ومراقبة كم تحرّكت `f`.

</div>

#### Reference Implementation

```python
def forward_circuit(x, y, z, w):
    p = x * y          # بوابة ضرب
    m = max(z, w)      # بوابة أعظم
    s = p + m          # بوابة جمع
    f = 2 * s          # ضرب بثابت
    return {'p': p, 'm': m, 's': s, 'f': f}

circuit_values = forward_circuit(3.0, -4.0, 2.0, -1.0)
print("Forward Pass:", circuit_values)

def sensitivity(x, y, z, w, h=1e-5):
    f_base  = forward_circuit(x, y, z, w)['f']
    f_moved = forward_circuit(x + h, y, z, w)['f']
    return (f_moved - f_base) / h

sensitivity_x = sensitivity(3.0, -4.0, 2.0, -1.0)
print(f"df/dx ≈ {sensitivity_x:.4f}")
```

#### Explanation

<div dir="rtl">

- `p = 3 × (-4) = -12` — بوابة الضرب.
- `m = max(2, -1) = 2` — البوابة تختار `z` لأنه الأكبر، و `w` "يخسر" تماماً.
- `s = -12 + 2 = -10` ثم `f = 2 × (-10) = -20`.
- الحساسية العددية تُعطي `≈ -8`: لو زدنا `x` بمقدار `h`، تنقص `f` بمقدار `8h`. لاحظ أن `-8 = 2 × y`، وهذا بالضبط ما ستحصل عليه تحليلياً بقاعدة السلسلة في التمرين القادم.

</div>

#### Expected Result

```
Forward Pass: {'p': -12.0, 'm': 2.0, 's': -10.0, 'f': -20.0}
df/dx ≈ -8.0000
```

#### Common Mistakes

<div dir="rtl">

- كتابة `f = 2 * (x*y + max(z,w))` في سطر واحد — تفقد القيم الوسيطة التي يحتاجها التمرير الخلفي.
- استخدام `h` كبيرة جداً (مثل `0.1`) فتكون المشتقة العددية غير دقيقة، أو صغيرة جداً (مثل `1e-15`) فتظهر أخطاء تقريب الفاصلة العائمة.
- كتابة أرقام ثابتة داخل الدالة بدل استخدام المعاملات `x, y, z, w` — الدالة عندها تفشل على أي مدخلات أخرى.

</div>

</details>

---

<div dir="rtl">
<h2>Exercise 2 — الانتشار الخلفي وقاعدة السلسلة | Backpropagation &amp; Chain Rule</h2>

<h3 style="direction: rtl;">🎯 Learning Objective</h3>
<p style="direction: rtl; text-align: right;">أن تنفّذ <strong>التمرير الخلفي (Backward Pass)</strong> يدوياً على نفس الدارة: تبدأ من الخرج بتدرّج <code>1.0</code>، وتضرب في <strong>التدرّج المحلي</strong> لكل بوابة، وتوزّع "اللوم" حتى تصل إلى المدخلات الأربعة.</p>

<h3 style="direction: rtl;">🧩 Concept</h3>
<p style="direction: rtl; text-align: right;"><strong>الانتشار الخلفي (Backpropagation)</strong> و <strong>قاعدة السلسلة (Chain Rule)</strong>:</p>
<p style="direction: rtl; text-align: center; font-size: 1.1em;">$$\frac{\partial f}{\partial x} = \frac{\partial f}{\partial p} \cdot \frac{\partial p}{\partial x}$$</p>
<p style="direction: rtl; text-align: right;">وسلوك البوابات في التمرير الخلفي: <strong>الجمع يوزّع</strong>، <strong>الضرب يبدّل</strong>، <strong>الأعظم يوجّه</strong>.</p>

<h3 style="direction: rtl;">📖 Scenario</h3>
<p style="direction: rtl; text-align: right;">عندما تكتب <code>loss.backward()</code> في PyTorch، فإن ما يحدث خلف الكواليس هو بالضبط ما ستكتبه الآن بيدك — لكن على ملايين الأوزان بدل أربعة مدخلات. من يفهم هذه الأسطر العشرة يفهم التعلم العميق كله.</p>

<h3 style="direction: rtl;">📝 Task</h3>
<p style="direction: rtl; text-align: right;">على نفس دارة التمرين الأول ($f = 2(x \cdot y + \max(z,w))$ مع $x=3,\ y=-4,\ z=2,\ w=-1$):</p>
<ol style="direction: rtl; text-align: right;">
<li>اكتب دالة <code>backward_circuit(x, y, z, w)</code> تُنفّذ <strong>التمرير الأمامي أولاً</strong> (لأنك تحتاج القيم الوسيطة)، ثم التمرير الخلفي.</li>
<li>ابدأ بالتدرّج عند الخرج: <code>df_df = 1.0</code>.</li>
<li>مرّر التدرّج عبر البوابات <strong>بالترتيب المعاكس</strong>:
   <ul style="direction: rtl; text-align: right; margin-top: 5px;">
   <li>بوابة <code>f = 2 * s</code> → التدرّج المحلي = 2</li>
   <li>بوابة <code>s = p + m</code> (<strong>جمع</strong>) → تُوزّع التدرّج كما هو على <code>p</code> و <code>m</code></li>
   <li>بوابة <code>p = x * y</code> (<strong>ضرب</strong>) → تُبدّل: مشتقة كل مدخل هي <strong>المدخل الآخر</strong></li>
   <li>بوابة <code>m = max(z, w)</code> (<strong>أعظم</strong>) → التدرّج كاملاً للفائز، و <strong>صفر</strong> للخاسر</li>
   </ul>
</li>
<li>أعِد <code>tuple</code> من عنصرين: <code>(f, grads)</code> حيث <code>grads</code> قاموس بالمفاتيح <code>'x'</code>, <code>'y'</code>, <code>'z'</code>, <code>'w'</code>.</li>
<li>خزّن الناتج في <code>f_value, grads</code> واطبع التدرجات الأربعة.</li>
<li>قارن <code>grads['x']</code> مع <code>sensitivity_x</code> من التمرين الأول.</li>
</ol>

<h3 style="direction: rtl;">Requirements</h3>
<ul style="direction: rtl; text-align: right;">
<li><strong>المكتبات:</strong> Python خالص — <strong>ممنوع</strong> استخدام <code>autograd</code> هنا.</li>
<li><strong>الدالة المطلوبة:</strong> <code>backward_circuit(x, y, z, w)</code> تُعيد <code>(f, grads)</code>.</li>
<li><strong>المتغيرات المطلوبة:</strong> <code>f_value</code> و <code>grads</code>.</li>
<li><strong>المخرجات:</strong> <code>grads</code> قاموس بأربعة مفاتيح: <code>x</code>, <code>y</code>, <code>z</code>, <code>w</code>، وكل قيمة رقم.</li>
<li><strong>قيود:</strong> يجب أن يمر التدرّج عبر البوابات خطوة بخطوة (ضرب التدرّج القادم في التدرّج المحلي) — لا تكتب النتيجة النهائية مباشرة.</li>
</ul>

<h3 style="direction: rtl;">Expected Result</h3>
<ul style="direction: rtl; text-align: right;">
<li>ستحصل على أربعة تدرجات: اثنان لهما إشارتان مختلفتان (من بوابة الضرب)، وواحد موجب من بوابة الـ max، و <strong>واحد يساوي صفراً بالضبط</strong>. هذا الصفر ليس خطأ — بل هو جوهر بوابة الـ <code>max</code>: المدخل الخاسر <strong>لا يؤثر إطلاقاً</strong> على الخرج، فلا يتحمّل أي "لوم". كذلك ستلاحظ أن <code>grads['x']</code> يطابق <code>sensitivity_x</code> العددي من التمرين الأول.</li>
</ul>

</div>

### 💡 Hints

<details>
<summary>💡 Hint 1</summary>

<div dir="rtl">

اقلب ترتيب أسطر التمرير الأمامي. آخر بوابة نفّذتها في الأمام هي **أول** بوابة تعالجها في الخلف. كل بوابة تستلم "التدرّج القادم من الأمام" وتضربه في تدرّجها المحلي.

</div>

</details>

<details>
<summary>💡 Hint 2</summary>

<div dir="rtl">

- بوابة الجمع `s = p + m`: التدرّج المحلي = 1 لكل مدخل، إذن `df_dp = df_ds * 1` و `df_dm = df_ds * 1`.
- بوابة الضرب `p = x * y`: `df_dx = df_dp * y` و `df_dy = df_dp * x` — **تبديل**!

</div>

</details>

<details>
<summary>💡 Hint 3</summary>

<div dir="rtl">

لبوابة الـ max استخدم شرطاً منطقياً:

```python
df_dz = df_dm * (1.0 if z >= w else 0.0)
df_dw = df_dm * (1.0 if w >  z else 0.0)
```

</div>

</details>

### 🧑‍💻 Your Solution

In [ ]:
# TODO: أكمل التنفيذ

def backward_circuit(x, y, z, w):
    # --- Forward Pass (نحتاج القيم الوسيطة) ---
    # p = ...
    # m = ...
    # s = ...
    # f = ...

    # --- Backward Pass (بالترتيب المعاكس) ---
    # df_df = 1.0
    # df_ds = ...      # بوابة الضرب بـ 2
    # df_dp, df_dm = ...  # بوابة الجمع: توزّع
    # df_dx, df_dy = ...  # بوابة الضرب: تبدّل
    # df_dz, df_dw = ...  # بوابة الأعظم: توجّه للفائز

    # return f, {'x': df_dx, 'y': df_dy, 'z': df_dz, 'w': df_dw}
    pass

# نفّذ على x=3, y=-4, z=2, w=-1 واطبع النتائج
f_value, grads = None, None

# Write your solution below

### ✅ Self-Check

In [ ]:
assert 'backward_circuit' in dir(), "الدالة 'backward_circuit' لم تُعرَّف."
assert 'grads' in dir() and grads is not None, "المتغير 'grads' لم يُنشأ."
assert isinstance(grads, dict), "يجب أن يكون 'grads' قاموساً (dict)."

for key in ['x', 'y', 'z', 'w']:
    assert key in grads, f"التدرّج بالنسبة لـ '{key}' مفقود."

assert abs(f_value - (-20)) < 1e-6, f"قيمة f خاطئة: {f_value} (المتوقع -20)."
assert abs(grads['x'] - (-8)) < 1e-6, f"df/dx خاطئ: {grads['x']} — تذكّر أن بوابة الضرب تُبدّل المدخلات."
assert abs(grads['y'] -   6)  < 1e-6, f"df/dy خاطئ: {grads['y']} (المتوقع 6)."
assert abs(grads['z'] -   2)  < 1e-6, f"df/dz خاطئ: {grads['z']} — بوابة max تُمرّر التدرّج كاملاً للفائز."
assert abs(grads['w'] -   0)  < 1e-9, f"df/dw خاطئ: {grads['w']} — المدخل الخاسر في بوابة max يأخذ صفراً."

# التحقق من العمومية: لو انقلب الفائز في بوابة max
_, g2 = backward_circuit(3.0, -4.0, -1.0, 2.0)
assert abs(g2['w'] - 2) < 1e-6 and abs(g2['z']) < 1e-9, \
    "بوابة max لا تتعامل مع تغيّر الفائز — تحقق من الشرط المنطقي."

print("✅ Basic checks passed.")
print(f"   grads = {grads}")

### 💡 Solution

<details>
<summary>💡 Show Solution</summary>

#### Approach

<div dir="rtl">

نُنفّذ التمرير الأمامي لتخزين القيم الوسيطة، ثم نعود من الخرج للمدخلات. كل بوابة تعرف شيئين فقط: قيمتها في الأمام، وتدرّجها المحلي. قاعدة السلسلة هي مجرد **ضرب** التدرّج القادم في التدرّج المحلي.

</div>

#### Reference Implementation

```python
def backward_circuit(x, y, z, w):
    # --- Forward Pass ---
    p = x * y
    m = max(z, w)
    s = p + m
    f = 2 * s

    # --- Backward Pass ---
    df_df = 1.0
    df_ds = df_df * 2.0                        # f = 2*s

    df_dp = df_ds * 1.0                        # بوابة جمع: توزّع
    df_dm = df_ds * 1.0

    df_dx = df_dp * y                          # بوابة ضرب: تبدّل
    df_dy = df_dp * x

    df_dz = df_dm * (1.0 if z >= w else 0.0)   # بوابة أعظم: توجّه
    df_dw = df_dm * (1.0 if w >  z else 0.0)

    return f, {'x': df_dx, 'y': df_dy, 'z': df_dz, 'w': df_dw}

f_value, grads = backward_circuit(3.0, -4.0, 2.0, -1.0)
print(f"f = {f_value}")
for k, v in grads.items():
    print(f"df/d{k} = {v}")
```

#### Explanation

<div dir="rtl">

- `df/dx = 2 × y = -8`: زيادة `x` تُنقص `f` — وهذا يطابق تماماً القياس العددي في التمرين الأول ✅
- `df/dy = 2 × x = 6`: زيادة `y` تزيد `f`.
- `df/dz = 2`: `z` هو الفائز في بوابة الـ max، فيأخذ التدرّج كاملاً.
- `df/dw = 0`: `w` خاسر — تغييره الطفيف لا يغيّر الخرج إطلاقاً، فلا يتحمّل أي مسؤولية.

</div>

#### Expected Result

```
f = -20.0
df/dx = -8.0
df/dy = 6.0
df/dz = 2.0
df/dw = 0.0
```

#### Common Mistakes

<div dir="rtl">

- عكس التبديل في بوابة الضرب: كتابة `df_dx = df_dp * x` بدل `* y`.
- نسيان ضرب التدرّج المحلي في **التدرّج القادم** (`df_ds`) والاكتفاء بالتدرّج المحلي وحده.
- إعطاء المدخل الخاسر في بوابة الـ max نصف التدرّج أو التدرّج كاملاً بدل الصفر.

</div>

</details>

---

<div dir="rtl">
<h2>Exercise 3 — التدرجات عند التفرّع والتحقق بـ Autograd | Gradients at Branches &amp; Autograd Check</h2>

<h3 style="direction: rtl;">🎯 Learning Objective</h3>
<p style="direction: rtl; text-align: right;">أن تحسب التدرجات في دارة <strong>يتفرّع فيها مدخل واحد إلى بوابتين</strong>، وتكتشف بنفسك قاعدة <strong>تجميع التدرجات عند التفرّع</strong>، ثم تتحقق من حسابك اليدوي عبر <code>autograd</code> في PyTorch.</p>

<h3 style="direction: rtl;">🧩 Concept</h3>
<p style="direction: rtl; text-align: right;"><strong>تجميع التدرجات (Gradient Accumulation)</strong> عند التفرّع، وتفسير التدرّج كـ <strong>حساسية</strong>، والربط بين الحساب اليدوي و <code>loss.backward()</code>.</p>

<h3 style="direction: rtl;">📖 Scenario</h3>
<p style="direction: rtl; text-align: right;">في الشبكات الحقيقية، مخرَج أي عصبون يذهب إلى <strong>عدة عصبونات</strong> في الطبقة التالية. فماذا يحدث للتدرّج عندما يتفرّع متغير إلى أكثر من مسار؟ هذا بالضبط ما يميّز الشبكة الحقيقية عن الدارة الخطية البسيطة — وهو تمرين رقم 4 في نهاية نوتبوك الأسبوع الخامس.</p>

<h3 style="direction: rtl;">📝 Task</h3>
<p style="direction: rtl; text-align: right;">خذ المعادلة:</p>
<p style="direction: rtl; text-align: center; font-size: 1.1em;">$$f = (a + b) \cdot (b + 1) \qquad \text{مع} \quad a = 2,\ b = 3$$</p>
<p style="direction: rtl; text-align: right;">انتبه: <strong><code>b</code> يظهر مرتين</strong> — يتفرّع إلى بوابتَي جمع مختلفتين.</p>
<ol style="direction: rtl; text-align: right;">
<li>اكتب دالة <code>manual_grads(a, b)</code> تُنفّذ التمرير الأمامي (<code>u = a + b</code>, <code>v = b + 1</code>, <code>f = u * v</code>) ثم التمرير الخلفي يدوياً.</li>
<li>عند حساب التدرّج بالنسبة لـ <code>b</code>، <strong>اجمع</strong> مساهمة المسارين.</li>
<li>أعِد <code>(f, df_da, df_db)</code> وخزّن النتيجة في <code>f_manual, grad_a_manual, grad_b_manual</code>.</li>
<li>أنشئ نفس المعادلة في PyTorch باستخدام <code>torch.tensor(..., requires_grad=True)</code> للمتغيرين <code>a_t</code> و <code>b_t</code>.</li>
<li>نفّذ <code>f_t.backward()</code> واقرأ <code>a_t.grad</code> و <code>b_t.grad</code>، وخزّنهما في <code>grad_a_torch</code> و <code>grad_b_torch</code>.</li>
<li>اطبع مقارنة بين الحساب اليدوي وحساب PyTorch.</li>
</ol>

<h3 style="direction: rtl;">Requirements</h3>
<ul style="direction: rtl; text-align: right;">
<li><strong>المكتبات:</strong> <code>torch</code>.</li>
<li><strong>الدالة المطلوبة:</strong> <code>manual_grads(a, b)</code> تُعيد <code>(f, df_da, df_db)</code>.</li>
<li><strong>المتغيرات المطلوبة:</strong> <code>f_manual</code>, <code>grad_a_manual</code>, <code>grad_b_manual</code>, <code>a_t</code>, <code>b_t</code>, <code>grad_a_torch</code>, <code>grad_b_torch</code>.</li>
<li><strong>المدخلات:</strong> <code>a = 2.0</code>, <code>b = 3.0</code> (استخدم <code>float</code> وليس <code>int</code> في PyTorch).</li>
<li><strong>قيود:</strong> في <code>manual_grads</code> ممنوع استخدام <code>autograd</code> — الحساب يدوي بالكامل. PyTorch يُستخدم <strong>فقط</strong> للتحقق.</li>
</ul>

<h3 style="direction: rtl;">Expected Result</h3>
<ul style="direction: rtl; text-align: right;">
<li>القيمتان يجب أن تتطابقا تماماً: التدرّج اليدوي بالنسبة لـ <code>b</code> = التدرّج الذي حسبه PyTorch. وستلاحظ أن التدرّج بالنسبة لـ <code>b</code> <strong>أكبر بكثير</strong> من التدرّج بالنسبة لـ <code>a</code> — لأن <code>b</code> يؤثّر على الخرج عبر <strong>مسارين</strong> لا مسار واحد. لو نسيت جمع المسارين، ستحصل على رقم أصغر ولن يتطابق مع PyTorch.</li>
</ul>

</div>

### 💡 Hints

<details>
<summary>💡 Hint 1</summary>

<div dir="rtl">

ارسم الدارة على ورقة أولاً: `a` يدخل بوابة جمع واحدة، بينما `b` يخرج منه سهمان — واحد إلى `u = a + b` وآخر إلى `v = b + 1`. كل سهم يعود بتدرّج مستقل.

</div>

</details>

<details>
<summary>💡 Hint 2</summary>

<div dir="rtl">

بوابة الضرب `f = u * v` تُعطي: `df_du = v` و `df_dv = u`. ثم بوابتا الجمع تُمرّران التدرّج كما هو (التدرّج المحلي = 1). التدرّج النهائي لـ `b` هو **مجموع** ما يصله من المسارين.

</div>

</details>

<details>
<summary>💡 Hint 3</summary>

<div dir="rtl">

```python
df_da = df_du * 1.0
df_db = df_du * 1.0 + df_dv * 1.0   # ← الجمع عند التفرّع!

# وفي PyTorch:
a_t = torch.tensor(2.0, requires_grad=True)
b_t = torch.tensor(3.0, requires_grad=True)
f_t = (a_t + b_t) * (b_t + 1)
f_t.backward()
```

</div>

</details>

### 🧑‍💻 Your Solution

In [ ]:
# TODO: أكمل التنفيذ

# Step 1: الحساب اليدوي
def manual_grads(a, b):
    # Forward:  u = a + b ,  v = b + 1 ,  f = u * v
    # Backward: df_du = ? , df_dv = ?
    #           df_da = ? , df_db = ?  (انتبه للتفرّع)
    pass

f_manual, grad_a_manual, grad_b_manual = None, None, None

# Step 2: التحقق باستخدام autograd
a_t = None
b_t = None
f_t = None
# f_t.backward()

grad_a_torch = None
grad_b_torch = None

# Step 3: اطبع المقارنة
# Write your solution below

### ✅ Self-Check

In [ ]:
assert 'manual_grads' in dir(), "الدالة 'manual_grads' لم تُعرَّف."
assert f_manual is not None and grad_b_manual is not None, "نتائج الحساب اليدوي لم تُخزَّن."

assert abs(f_manual - 20) < 1e-6, f"قيمة f خاطئة: {f_manual} (المتوقع 20)."
assert abs(grad_a_manual - 4) < 1e-6, f"df/da خاطئ: {grad_a_manual} (المتوقع 4)."
assert abs(grad_b_manual - 9) < 1e-6, (
    f"df/db خاطئ: {grad_b_manual}. تذكّر: b يتفرّع إلى بوابتين، والتدرجات تُجمع عند التفرّع!")

assert isinstance(a_t, torch.Tensor) and a_t.requires_grad, "a_t ليس tensor بـ requires_grad=True."
assert isinstance(b_t, torch.Tensor) and b_t.requires_grad, "b_t ليس tensor بـ requires_grad=True."
assert grad_a_torch is not None and grad_b_torch is not None, \
    "لم تقرأ .grad — هل نسيت استدعاء f_t.backward()؟"

assert abs(float(grad_a_torch) - grad_a_manual) < 1e-5, "الحساب اليدوي لا يطابق autograd عند a."
assert abs(float(grad_b_torch) - grad_b_manual) < 1e-5, "الحساب اليدوي لا يطابق autograd عند b."

# التحقق من العمومية على قيم أخرى
f2, ga2, gb2 = manual_grads(1.0, 1.0)
assert abs(f2 - 4) < 1e-6 and abs(ga2 - 2) < 1e-6 and abs(gb2 - 4) < 1e-6, \
    "الدالة لا تعمل على مدخلات أخرى — تأكد أنك تستخدم المعاملات."

print("✅ Basic checks passed.")
print(f"   manual: df/da={grad_a_manual}, df/db={grad_b_manual}")
print(f"   torch : df/da={float(grad_a_torch)}, df/db={float(grad_b_torch)}")

### 💡 Solution

<details>
<summary>💡 Show Solution</summary>

#### Approach

<div dir="rtl">

نفكّك المعادلة إلى ثلاث بوابات، ونعود من الخرج. المتغيّر `b` يصله تدرّج من مسارين مختلفين، فنجمعهما. ثم نبني نفس المعادلة في PyTorch ونترك `autograd` يقوم بالعمل — النتيجتان يجب أن تتطابقا.

</div>

#### Reference Implementation

```python
def manual_grads(a, b):
    # --- Forward ---
    u = a + b          # بوابة جمع
    v = b + 1          # بوابة جمع
    f = u * v          # بوابة ضرب

    # --- Backward ---
    df_du = v          # بوابة ضرب: تبدّل
    df_dv = u

    df_da = df_du * 1.0
    df_db = df_du * 1.0 + df_dv * 1.0   # ← تجميع التدرجات عند التفرّع
    return f, df_da, df_db

f_manual, grad_a_manual, grad_b_manual = manual_grads(2.0, 3.0)

# --- التحقق بـ autograd ---
a_t = torch.tensor(2.0, requires_grad=True)
b_t = torch.tensor(3.0, requires_grad=True)
f_t = (a_t + b_t) * (b_t + 1)
f_t.backward()

grad_a_torch = a_t.grad.item()
grad_b_torch = b_t.grad.item()

print(f"يدوي   : f={f_manual}, df/da={grad_a_manual}, df/db={grad_b_manual}")
print(f"PyTorch: f={f_t.item()}, df/da={grad_a_torch}, df/db={grad_b_torch}")
```

#### Explanation

<div dir="rtl">

- `u = 5`, `v = 4`, `f = 20`.
- `df/du = v = 4` و `df/dv = u = 5`.
- `df/da = 4` — مسار واحد فقط.
- `df/db = 4 + 5 = 9` — **مجموع** المسارين. هذا هو السبب الذي يجعل PyTorch **يُراكم** التدرجات في `.grad`، ولهذا نكتب `optimizer.zero_grad()` في بداية كل خطوة تدريب — وإلا تراكمت تدرجات الدورة السابقة فوق الحالية!

</div>

#### Expected Result

```
يدوي   : f=20.0, df/da=4.0, df/db=9.0
PyTorch: f=20.0, df/da=4.0, df/db=9.0
```

#### Common Mistakes

<div dir="rtl">

- حساب `df/db` من مسار واحد فقط (النتيجة 4 أو 5 بدل 9).
- نسيان `requires_grad=True` فيكون `a_t.grad` قيمته `None`.
- استدعاء `.backward()` أكثر من مرة على نفس الرسم البياني فتظهر رسالة خطأ، أو تراكم التدرجات دون تصفيرها بـ `.grad = None`.

</div>

</details>

---

<div dir="rtl">
<h2>Exercise 4 — دوال الخسارة MSE و Cross-Entropy | Loss Functions</h2>

<h3 style="direction: rtl;">🎯 Learning Objective</h3>
<p style="direction: rtl; text-align: right;">أن تنفّذ <strong>دالة الخسارة</strong> بنفسك — <code>MSE</code> للانحدار و <code>Cross-Entropy</code> للتصنيف — وتتحقق من مطابقتها لدوال PyTorch الجاهزة، وتلاحظ كيف تعاقب Cross-Entropy الثقة العمياء الخاطئة.</p>

<h3 style="direction: rtl;">🧩 Concept</h3>
<p style="direction: rtl; text-align: right;"><strong>دالة الخسارة (Loss Function)</strong>، <strong>متوسط مربعات الخطأ (MSE)</strong>، <strong>الإنتروبيا المتقاطعة (Cross-Entropy)</strong>، <strong>Softmax</strong>، والفرق بين <code>Error</code> و <code>Loss</code> و <code>Cost</code>:</p>
<p style="direction: rtl; text-align: center; font-size: 1.1em;">$$L_{MSE} = \frac{1}{N}\sum_{i=1}^{N}(\hat{y}_i - y_i)^2 \qquad\qquad L_{CE} = -\log(p_{\text{correct class}})$$</p>

<h3 style="direction: rtl;">📖 Scenario</h3>
<p style="direction: rtl; text-align: right;">الخسارة هي "المصبّ" الذي تبدأ منه رحلة الانتشار الخلفي. اختيار الدالة الخاطئة يعني إشارة تدريب خاطئة. في MNIST نحن أمام مسألة <strong>تصنيف</strong> (10 فئات)، ولذلك نستخدم <code>CrossEntropyLoss</code> وليس <code>MSELoss</code> — لكن لن تصدّق ذلك حتى تحسبهما بنفسك.</p>

<h3 style="direction: rtl;">📝 Task</h3>
<ol style="direction: rtl; text-align: right;">
<li><strong>MSE يدوياً:</strong> أنشئ <code>y_true = torch.tensor([3.0, -0.5, 2.0, 7.0])</code> و <code>y_pred = torch.tensor([2.5, 0.0, 2.0, 8.0])</code>، واحسب <code>mse_manual</code> بصيغة متوسط مربعات الفروق (بدون استخدام <code>nn.MSELoss</code>).</li>
<li>احسب <code>mse_torch</code> باستخدام <code>nn.MSELoss()</code> وقارن.</li>
<li><strong>Cross-Entropy يدوياً:</strong> استخدم مصفوفة الـ logits التالية لصورتين من MNIST:<br>
<pre style="direction: ltr; text-align: left; background-color: #f5f5f5; padding: 8px; border-radius: 5px; margin: 5px 0;">
logits = torch.tensor([[2.0, 1.0, 0.1, 0.0, 0.0, 0.0, 0.0, 5.0, 0.0, 0.0],
                       [0.5, 0.2, 3.0, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]])
labels = torch.tensor([7, 2])
</pre>
</li>
<li>حوّل الـ logits إلى احتمالات باستخدام <code>torch.softmax(logits, dim=1)</code> وخزّنها في <code>probs</code>.</li>
<li>احسب <code>ce_manual</code> = متوسط $-\log(p_{\text{الفئة الصحيحة}})$ للصورتين، ثم <code>ce_torch</code> باستخدام <code>nn.CrossEntropyLoss()</code> وقارن.</li>
<li><strong>تجربة العقوبة:</strong> احسب <code>loss_right = -log(0.99)</code> و <code>loss_wrong = -log(0.001)</code> واطبع الفرق بينهما.</li>
</ol>

<h3 style="direction: rtl;">Requirements</h3>
<ul style="direction: rtl; text-align: right;">
<li><strong>المكتبات:</strong> <code>torch</code>, <code>torch.nn as nn</code>.</li>
<li><strong>المتغيرات المطلوبة:</strong> <code>mse_manual</code>, <code>mse_torch</code>, <code>probs</code>, <code>ce_manual</code>, <code>ce_torch</code>, <code>loss_right</code>, <code>loss_wrong</code>.</li>
<li><strong>الأبعاد:</strong> <code>logits.shape == (2, 10)</code> و <code>labels.shape == (2,)</code> و <code>probs.shape == (2, 10)</code>.</li>
<li><strong>قيود:</strong> في الحساب اليدوي استخدم عمليات tensor الأساسية فقط (<code>**</code>, <code>.mean()</code>, <code>torch.log</code>, <code>torch.softmax</code>) — ممنوع استدعاء <code>nn.MSELoss</code> أو <code>nn.CrossEntropyLoss</code> داخل الحساب اليدوي.</li>
<li><strong>ملاحظة مهمة:</strong> <code>nn.CrossEntropyLoss</code> في PyTorch تُطبّق <code>softmax</code> داخلياً، لذلك تُمرَّر لها <strong>logits خام</strong> وليست احتمالات.</li>
</ul>

<h3 style="direction: rtl;">Expected Result</h3>
<ul style="direction: rtl; text-align: right;">
<li>القيمة اليدوية والقيمة من PyTorch يجب أن تتطابقا في الحالتين (بفارق أقل من <code>1e-6</code>). وستلاحظ أن مجموع كل صف في <code>probs</code> يساوي 1 بالضبط. أما تجربة العقوبة فستُظهر فرقاً <strong>هائلاً</strong> بين الخسارتين — الثقة العمياء في الجواب الخاطئ تكلّف الشبكة أضعافاً مضاعفة، وهذا بالضبط سبب اختيار Cross-Entropy لمسائل التصنيف.</li>
</ul>

</div>

### 💡 Hints

<details>
<summary>💡 Hint 1</summary>

<div dir="rtl">

MSE في سطر واحد: `((y_pred - y_true) ** 2).mean()`. لاحظ أن التربيع يُلغي الإشارات السالبة ويعاقب الأخطاء الكبيرة بشدة.

</div>

</details>

<details>
<summary>💡 Hint 2</summary>

<div dir="rtl">

لاختيار احتمال الفئة الصحيحة من كل صف، استخدم الفهرسة المتقدّمة:

```python
correct_probs = probs[torch.arange(len(labels)), labels]
```

هذا يأخذ العنصر رقم `labels[0]` من الصف 0، والعنصر رقم `labels[1]` من الصف 1.

</div>

</details>

<details>
<summary>💡 Hint 3</summary>

<div dir="rtl">

```python
probs = torch.softmax(logits, dim=1)
correct_probs = probs[torch.arange(len(labels)), labels]
ce_manual = -torch.log(correct_probs).mean()
ce_torch  = nn.CrossEntropyLoss()(logits, labels)   # logits خام، بدون softmax!
```

</div>

</details>

### 🧑‍💻 Your Solution

In [ ]:
# TODO: أكمل التنفيذ

# ===== Part 1: MSE =====
y_true = torch.tensor([3.0, -0.5, 2.0, 7.0])
y_pred = torch.tensor([2.5,  0.0, 2.0, 8.0])

mse_manual = None   # احسبها يدوياً
mse_torch  = None   # باستخدام nn.MSELoss

# ===== Part 2: Cross-Entropy =====
logits = torch.tensor([[2.0, 1.0, 0.1, 0.0, 0.0, 0.0, 0.0, 5.0, 0.0, 0.0],
                       [0.5, 0.2, 3.0, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]])
labels = torch.tensor([7, 2])

probs     = None   # softmax على البعد الصحيح
ce_manual = None   # متوسط -log(p للفئة الصحيحة)
ce_torch  = None   # باستخدام nn.CrossEntropyLoss

# ===== Part 3: عقوبة الثقة الخاطئة =====
loss_right = None  # -log(0.99)
loss_wrong = None  # -log(0.001)

# Write your solution below

### ✅ Self-Check

In [ ]:
import math

assert mse_manual is not None and mse_torch is not None, "قيم MSE لم تُحسب."
assert abs(float(mse_manual) - float(mse_torch)) < 1e-6, \
    f"MSE اليدوي ({float(mse_manual):.6f}) لا يطابق nn.MSELoss ({float(mse_torch):.6f})."
assert abs(float(mse_manual) - 0.375) < 1e-6, f"قيمة MSE خاطئة: {float(mse_manual)} (المتوقع 0.375)."

assert probs is not None, "المتغير 'probs' لم يُنشأ."
assert tuple(probs.shape) == (2, 10), f"شكل probs خاطئ: {tuple(probs.shape)} (المتوقع (2, 10))."
assert torch.allclose(probs.sum(dim=1), torch.ones(2), atol=1e-6), \
    "مجموع الاحتمالات في كل صف يجب أن يساوي 1 — تأكد أنك استخدمت dim=1 في softmax."

assert ce_manual is not None and ce_torch is not None, "قيم Cross-Entropy لم تُحسب."
assert abs(float(ce_manual) - float(ce_torch)) < 1e-5, (
    f"CE اليدوي ({float(ce_manual):.6f}) لا يطابق nn.CrossEntropyLoss ({float(ce_torch):.6f}). "
    "تذكّر: nn.CrossEntropyLoss تأخذ logits خام وليست احتمالات.")

assert loss_right is not None and loss_wrong is not None, "قيم تجربة العقوبة لم تُحسب."
assert abs(float(loss_right) - 0.01005) < 1e-3, "قيمة -log(0.99) خاطئة."
assert float(loss_wrong) > 50 * float(loss_right), \
    "الثقة الخاطئة يجب أن تُنتج خسارة أكبر بعشرات المرات."

print("✅ Basic checks passed.")
print(f"   MSE = {float(mse_manual):.4f} | CE = {float(ce_manual):.4f}")
print(f"   -log(0.99) = {float(loss_right):.4f}  vs  -log(0.001) = {float(loss_wrong):.4f}")

### 💡 Solution

<details>
<summary>💡 Show Solution</summary>

#### Approach

<div dir="rtl">

نُنفّذ كل دالة خسارة بصيغتها الرياضية المباشرة، ثم نستدعي نسخة PyTorch الجاهزة على نفس البيانات ونتأكد من التطابق. هذا يثبت أن `nn.CrossEntropyLoss` ليست صندوقاً أسود — بل `softmax` متبوعة بـ `-log` للفئة الصحيحة ثم متوسط.

</div>

#### Reference Implementation

```python
# ===== Part 1: MSE =====
y_true = torch.tensor([3.0, -0.5, 2.0, 7.0])
y_pred = torch.tensor([2.5,  0.0, 2.0, 8.0])

mse_manual = ((y_pred - y_true) ** 2).mean()
mse_torch  = nn.MSELoss()(y_pred, y_true)
print(f"MSE manual = {mse_manual.item():.6f} | MSE torch = {mse_torch.item():.6f}")

# ===== Part 2: Cross-Entropy =====
logits = torch.tensor([[2.0, 1.0, 0.1, 0.0, 0.0, 0.0, 0.0, 5.0, 0.0, 0.0],
                       [0.5, 0.2, 3.0, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]])
labels = torch.tensor([7, 2])

probs = torch.softmax(logits, dim=1)
correct_probs = probs[torch.arange(len(labels)), labels]
ce_manual = -torch.log(correct_probs).mean()
ce_torch  = nn.CrossEntropyLoss()(logits, labels)
print(f"CE manual = {ce_manual.item():.6f} | CE torch = {ce_torch.item():.6f}")

# ===== Part 3: عقوبة الثقة الخاطئة =====
loss_right = -torch.log(torch.tensor(0.99))
loss_wrong = -torch.log(torch.tensor(0.001))
print(f"واثقة وصائبة : {loss_right.item():.4f}")
print(f"واثقة ومخطئة : {loss_wrong.item():.4f}  💥")
```

#### Explanation

<div dir="rtl">

- **MSE**: التربيع يمنع الأخطاء الموجبة والسالبة من إلغاء بعضها، ويعاقب الخطأ الكبير أكثر من الصغير.
- **Cross-Entropy**: `softmax` تحوّل الـ logits إلى احتمالات مجموعها 1، ثم نأخذ `-log` لاحتمال الفئة الصحيحة فقط. الصورة الأولى (label=7) توقّعها قوي فخسارتها صغيرة، بينما الثانية أقل ثقة فخسارتها أكبر.
- **العقوبة**: `-log(0.99) ≈ 0.01` بينما `-log(0.001) ≈ 6.91` — فرق يزيد عن **680 ضعفاً**. الدالة لا تعاقب الخطأ فحسب، بل تعاقب **الثقة العمياء في الجواب الخاطئ**.

</div>

#### Expected Result

```
MSE manual = 0.375000 | MSE torch = 0.375000
CE manual = 0.256477 | CE torch = 0.256477
واثقة وصائبة : 0.0101
واثقة ومخطئة : 6.9078  💥
```

#### Common Mistakes

<div dir="rtl">

- تمرير `probs` بدل `logits` إلى `nn.CrossEntropyLoss` — النتيجة خاطئة لأن الدالة تُطبّق `softmax` مرة أخرى.
- استخدام `dim=0` بدل `dim=1` في `softmax` فيصبح التطبيع عبر الصور بدل الفئات.
- استخدام `.sum()` بدل `.mean()` في MSE أو CE — تحصل على `Cost` غير مقسوم على عدد العينات.

</div>

</details>

---

<div dir="rtl">
<h2>Exercise 5 — الانحدار التدرّجي ومعدل التعلم | Gradient Descent &amp; Learning Rate</h2>

<h3 style="direction: rtl;">🎯 Learning Objective</h3>
<p style="direction: rtl; text-align: right;">أن تنفّذ <strong>قاعدة تحديث المُحسّن (Optimizer Update Rule)</strong> بيدك على دالة خسارة بسيطة، وتقيس أثر <strong>معدل التعلم (Learning Rate)</strong> على مسار الوصول إلى القاع.</p>

<h3 style="direction: rtl;">🧩 Concept</h3>
<p style="direction: rtl; text-align: right;"><strong>المُحسّن (Optimizer)</strong> و <strong>الانحدار التدرّجي (Gradient Descent)</strong>:</p>
<p style="direction: rtl; text-align: center; font-size: 1.1em;">$$w \leftarrow w - \eta \cdot \frac{\partial L}{\partial w}$$</p>
<p style="direction: rtl; text-align: right;">مع دالة الخسارة $L(w) = w^2$ ومشتقتها $\frac{dL}{dw} = 2w$.</p>

<h3 style="direction: rtl;">📖 Scenario</h3>
<p style="direction: rtl; text-align: right;">الانتشار الخلفي يُخبرنا <strong>من المسؤول عن الخطأ</strong>، لكن المُحسّن هو من <strong>يعدّل الأوزان فعلياً</strong>. وحجم الخطوة $\eta$ يحدد كل شيء: صغير جداً = بطء قاتل، كبير جداً = انفجار. سنجرّب الثلاث حالات بأنفسنا — تخيّل كرة تتدحرج في وادٍ تريد الوصول لقاعه.</p>

<h3 style="direction: rtl;">📝 Task</h3>
<ol style="direction: rtl; text-align: right;">
<li>اكتب دالة <code>gradient_descent_path(lr, w0=4.0, steps=12)</code> تُنفّذ الانحدار التدرّجي على $L(w) = w^2$:
   <ul style="direction: rtl; text-align: right; margin-top: 5px;">
   <li>ابدأ من <code>w = w0</code> وخزّنها كأول عنصر في قائمة <code>path</code>.</li>
   <li>في كل خطوة: احسب التدرّج <code>grad = 2 * w</code>، ثم طبّق <code>w = w - lr * grad</code>، ثم أضف <code>w</code> إلى <code>path</code>.</li>
   <li>أعِد <code>path</code> (قائمة طولها <code>steps + 1</code>).</li>
   </ul>
</li>
<li>نفّذ الدالة بثلاث قيم لمعدل التعلم: <code>0.05</code> و <code>0.30</code> و <code>1.05</code>، وخزّن النتائج في قاموس اسمه <code>paths</code> بالمفاتيح <code>0.05, 0.3, 1.05</code>.</li>
<li>اطبع القيمة النهائية <code>path[-1]</code> لكل حالة.</li>
<li>ارسم الحالات الثلاث فوق منحنى $L(w) = w^2$ في ثلاثة رسوم متجاورة باستخدام <code>matplotlib</code> (استخدم <code>plt.subplots(1, 3, figsize=(15, 4.2))</code>).</li>
</ol>

<h3 style="direction: rtl;">Requirements</h3>
<ul style="direction: rtl; text-align: right;">
<li><strong>المكتبات:</strong> <code>numpy</code>, <code>matplotlib.pyplot</code>.</li>
<li><strong>الدالة المطلوبة:</strong> <code>gradient_descent_path(lr, w0=4.0, steps=12)</code>.</li>
<li><strong>المتغيرات المطلوبة:</strong> <code>paths</code> (dict بثلاثة مفاتيح).</li>
<li><strong>المخرجات:</strong> كل <code>path</code> قائمة (list) طولها 13 عنصراً.</li>
<li><strong>قيود:</strong> ممنوع استخدام <code>torch.optim</code> — نريد كتابة قاعدة التحديث بأيدينا سطراً بسطر.</li>
</ul>

<h3 style="direction: rtl;">Expected Result</h3>
<ul style="direction: rtl; text-align: right;">
<li>مع <code>lr = 0.05</code>: الكرة تقترب من القاع لكنها <strong>لم تصل بعد</strong> بعد 12 خطوة.</li>
<li>مع <code>lr = 0.30</code>: تصل إلى القاع (قيمة قريبة جداً من الصفر) بسرعة ✅</li>
<li>مع <code>lr = 1.05</code>: القيمة <strong>تكبر</strong> بدل أن تصغر — الكرة تقفز خارج الوادي وتنفجر 💥</li>
</ul>
<p style="direction: rtl; text-align: right;">في الرسم البياني سترى النقاط تتقارب نحو الصفر في الحالتين الأوليين، وتتباعد للخارج في الثالثة.</p>

</div>

### 💡 Hints

<details>
<summary>💡 Hint 1</summary>

<div dir="rtl">

القاعدة كلها سطر واحد داخل حلقة `for`: `w = w - lr * (2 * w)`. لا تنسَ إضافة القيمة الجديدة إلى القائمة بعد كل تحديث.

</div>

</details>

<details>
<summary>💡 Hint 2</summary>

<div dir="rtl">

لماذا ينفجر `lr = 1.05`؟ لأن `w_new = w - 1.05 × 2w = -1.1w`. كل خطوة تضرب القيمة في `-1.1` — تكبر وتتأرجح في الإشارة. جرّب طباعة القيم لترى التذبذب.

</div>

</details>

<details>
<summary>💡 Hint 3</summary>

<div dir="rtl">

للرسم:

```python
w_axis = np.linspace(-6, 6, 200)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for axi, lr in zip(axes, [0.05, 0.3, 1.05]):
    p = np.array(paths[lr])
    axi.plot(w_axis, w_axis**2, color='#94a3b8', lw=2)
    axi.plot(p, p**2, 'o-', color='#dc2626', markersize=6)
    axi.set_title(f'lr = {lr}')
```

</div>

</details>

### 🧑‍💻 Your Solution

In [ ]:
# TODO: أكمل التنفيذ

# Step 1: دالة الانحدار التدرّجي على L(w) = w²
def gradient_descent_path(lr, w0=4.0, steps=12):
    path = [w0]
    w = w0
    for _ in range(steps):
        # grad = ?
        # w = ?
        # path.append(w)
        pass
    return path

# Step 2: نفّذها بثلاثة معدلات تعلم
paths = {}

# Step 3: اطبع القيمة النهائية لكل حالة

# Step 4: ارسم الحالات الثلاث فوق منحنى الخسارة
# Write your solution below

### ✅ Self-Check

In [ ]:
assert 'gradient_descent_path' in dir(), "الدالة 'gradient_descent_path' لم تُعرَّف."
assert 'paths' in dir() and isinstance(paths, dict), "المتغير 'paths' لم يُنشأ كقاموس."

for lr in [0.05, 0.3, 1.05]:
    assert lr in paths, f"معدل التعلم {lr} مفقود من القاموس 'paths'."
    assert len(paths[lr]) == 13, f"طول المسار لـ lr={lr} خاطئ: {len(paths[lr])} (المتوقع 13 = steps+1)."
    assert abs(paths[lr][0] - 4.0) < 1e-9, "المسار يجب أن يبدأ من w0 = 4.0."

slow, good, boom = paths[0.05][-1], paths[0.3][-1], paths[1.05][-1]

assert abs(good) < 0.01, f"مع lr=0.3 يجب أن نصل للقاع تقريباً، لكن w النهائية = {good}."
assert abs(slow) > 0.5,  f"مع lr=0.05 يجب أن نبقى بعيدين عن القاع، لكن w النهائية = {slow}."
assert abs(boom) > 4.0,  f"مع lr=1.05 يجب أن تنفجر القيمة، لكن w النهائية = {boom}."

# التحقق من أن التحديث يستخدم المشتقة الصحيحة 2w
one_step = gradient_descent_path(0.1, w0=1.0, steps=1)
assert abs(one_step[1] - 0.8) < 1e-9, \
    f"خطوة واحدة من w=1 بـ lr=0.1 يجب أن تعطي 0.8 (لأن dL/dw = 2w)، لكنها أعطت {one_step[1]}."

print("✅ Basic checks passed.")
print(f"   lr=0.05 → w={slow:.4f} | lr=0.3 → w={good:.6f} | lr=1.05 → w={boom:.2f}")

### 💡 Solution

<details>
<summary>💡 Show Solution</summary>

#### Approach

<div dir="rtl">

نُنفّذ قاعدة التحديث حرفياً كما في المعادلة: احسب التدرّج عند الموضع الحالي، تحرّك **عكس** اتجاهه بمقدار `lr`، كرّر. نخزّن كل المواضع لنرسم المسار ونرى سلوك كل معدل تعلم بأعيننا.

</div>

#### Reference Implementation

```python
def gradient_descent_path(lr, w0=4.0, steps=12):
    path = [w0]
    w = w0
    for _ in range(steps):
        grad = 2 * w          # dL/dw حيث L = w²
        w = w - lr * grad     # قاعدة التحديث: w ← w - η·dL/dw
        path.append(w)
    return path

paths = {lr: gradient_descent_path(lr) for lr in [0.05, 0.3, 1.05]}

for lr, p in paths.items():
    print(f"lr = {lr:<5} → w النهائية = {p[-1]:.6f}")

# --- الرسم ---
w_axis = np.linspace(-6, 6, 200)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
titles = [ar('η صغير جداً: بطيء'), ar('η مناسب: يصل بسرعة'), ar('η كبير جداً: ينفجر!')]

for axi, lr, title in zip(axes, [0.05, 0.3, 1.05], titles):
    p = np.array(paths[lr])
    axi.plot(w_axis, w_axis**2, color='#94a3b8', lw=2)
    axi.plot(p, p**2, 'o-', color='#dc2626', markersize=6, lw=1.5, alpha=0.85)
    axi.plot(p[0], p[0]**2, 'o', color='#2563eb', markersize=10)
    axi.set_title(f'{title}\nlr = {lr}')
    axi.set_xlabel('w'); axi.set_ylabel('Loss')
    axi.set_xlim(-6.5, 6.5); axi.set_ylim(-2, 40)
    axi.grid(alpha=0.3)

plt.suptitle(ar('الانحدار التدرّجي: أثر معدل التعلم (النقطة الزرقاء = البداية)'), fontsize=13)
plt.tight_layout()
plt.show()
```

#### Explanation

<div dir="rtl">

- كل خطوة تضرب `w` في المعامل `(1 - 2·lr)`.
- `lr = 0.05` → المعامل `0.9`: انكماش بطيء جداً، 12 خطوة لا تكفي.
- `lr = 0.30` → المعامل `0.4`: انكماش سريع نحو الصفر ✅
- `lr = 1.05` → المعامل `-1.1`: القيمة تكبر وتتأرجح في الإشارة — **تباعد (Divergence)**.

هذا بالضبط ما يحدث في شبكة حقيقية عندما تختار `lr` كبيراً: الخسارة تقفز إلى `NaN` بدل أن تنخفض.

</div>

#### Expected Result

```
lr = 0.05  → w النهائية = 1.129717
lr = 0.3   → w النهائية = 0.000067
lr = 1.05  → w النهائية = 12.553713
```

#### Common Mistakes

<div dir="rtl">

- استخدام `w` بدل `2*w` كتدرّج (نسيان مشتقة التربيع).
- الجمع بدل الطرح: `w = w + lr * grad` — تصعد التلة بدل النزول!
- تحديث `w` داخل الحلقة بدون إضافتها إلى `path`، فيصبح طول القائمة خاطئاً.

</div>

</details>

---

<div dir="rtl">
<h2>Exercise 6 — تجهيز بيانات MNIST وبناء الشبكة | MNIST Data Pipeline &amp; FC Model</h2>

<h3 style="direction: rtl;">🎯 Learning Objective</h3>
<p style="direction: rtl; text-align: right;">أن تُجهّز مجموعة <strong>MNIST</strong> عبر <code>torchvision</code> مع <strong>التطبيع (Normalization)</strong> و <strong>الدفعات (Batches)</strong>، وتفحص أبعاد البيانات بنفسك، ثم تبني شبكة <strong>Fully Connected</strong> وتحسب عدد أوزانها.</p>

<h3 style="direction: rtl;">🧩 Concept</h3>
<p style="direction: rtl; text-align: right;"><strong>MNIST</strong>, <strong>Transforms &amp; Normalize</strong>, <strong>DataLoader &amp; Batch</strong>, <strong>Flatten</strong>, بنية <strong>Fully Connected Network</strong>, <strong>عدد المعاملات (Parameters)</strong>.</p>

<h3 style="direction: rtl;">📖 Scenario</h3>
<p style="direction: rtl; text-align: right;">MNIST هي "اختبار الذكاء" الأول لأي شبكة رؤية حاسوبية: 60,000 صورة تدريب و 10,000 صورة اختبار، كل صورة <strong>28×28</strong> بكسل رمادية لرقم من 0 إلى 9. قبل أن ندرّب أي نموذج، يجب أن نعرف بالضبط <strong>ما شكل البيانات الداخلة إليه</strong> — وهذه العادة ستنقذك من ساعات تصحيح أخطاء الأبعاد لاحقاً.</p>

<div align="center">
<img src="media/00-pixel-grid-data-matrix.png" width="620">
</div>

<h3 style="direction: rtl;">📝 Task</h3>
<ol style="direction: rtl; text-align: right;">
<li>أنشئ <code>transform</code> باستخدام <code>transforms.Compose</code> يحتوي على <code>ToTensor()</code> ثم <code>Normalize((0.5,), (0.5,))</code>.</li>
<li>حمّل <code>train_dataset</code> و <code>test_dataset</code> من <code>torchvision.datasets.MNIST</code> مع <code>root='./data'</code> و <code>download=True</code> و <code>transform=transform</code>.</li>
<li>أنشئ <code>train_loader</code> و <code>test_loader</code> بـ <code>batch_size=64</code>، مع <code>shuffle=True</code> للتدريب و <code>shuffle=False</code> للاختبار.</li>
<li>اسحب أول دفعة: <code>images, labels = next(iter(train_loader))</code> واطبع <code>images.shape</code> و <code>labels.shape</code> وقيمتَي <code>images.min()</code> و <code>images.max()</code>.</li>
<li>اعرض 10 صور من الدفعة في شبكة <code>2×5</code> باستخدام <code>matplotlib</code> مع عنوان كل صورة = الـ label الخاص بها.</li>
<li>ابنِ <code>fc_model</code> بـ <code>nn.Sequential</code> بالبنية التالية بالضبط:<br>
   <code>Flatten → Linear(784, 128) → ReLU → Linear(128, 64) → ReLU → Linear(64, 10)</code></li>
<li>احسب عدد الأوزان <code>fc_params = sum(p.numel() for p in fc_model.parameters())</code> واطبعه.</li>
</ol>

<h3 style="direction: rtl;">Requirements</h3>
<ul style="direction: rtl; text-align: right;">
<li><strong>المكتبات:</strong> <code>torch</code>, <code>torch.nn as nn</code>, <code>torchvision</code>, <code>torchvision.transforms as transforms</code>, <code>matplotlib.pyplot</code>.</li>
<li><strong>المتغيرات المطلوبة:</strong> <code>transform</code>, <code>train_dataset</code>, <code>test_dataset</code>, <code>train_loader</code>, <code>test_loader</code>, <code>images</code>, <code>labels</code>, <code>fc_model</code>, <code>fc_params</code>.</li>
<li><strong>الأبعاد المتوقعة:</strong> <code>images.shape == (64, 1, 28, 28)</code> و <code>labels.shape == (64,)</code>.</li>
<li><strong>قيود:</strong> لا تُغيّر البنية المطلوبة للشبكة — سنقارنها لاحقاً بشبكة CNN، والمقارنة يجب أن تكون عادلة.</li>
</ul>

<h3 style="direction: rtl;">Expected Result</h3>
<ul style="direction: rtl; text-align: right;">
<li>ستطبع الأبعاد: دفعة من <strong>64 صورة</strong>، كل صورة بقناة واحدة (grayscale) بحجم 28×28. بعد التطبيع، قيم البكسلات ستكون في مجال يمتد من <strong>سالب واحد إلى موجب واحد</strong> تقريباً بدل 0 إلى 1. وسترى شبكة من عشر صور لأرقام مكتوبة بخط اليد — بعضها واضح وبعضها يصعب حتى على الإنسان! أما عدد أوزان الشبكة فسيتجاوز <strong>مئة ألف وزن</strong> — احفظ هذا الرقم، سنقارنه بـ CNN في التمرين العاشر.</li>
</ul>

</div>

### 💡 Hints

<details>
<summary>💡 Hint 1</summary>

<div dir="rtl">

`Normalize((0.5,), (0.5,))` تعمل بالمعادلة `(x - mean) / std`. بما أن `ToTensor()` تُنتج قيماً بين 0 و 1، فإن النتيجة تصبح بين `(0-0.5)/0.5 = -1` و `(1-0.5)/0.5 = 1`. الأقواس بها فاصلة `(0.5,)` لأنها **tuple** لقناة واحدة.

</div>

</details>

<details>
<summary>💡 Hint 2</summary>

<div dir="rtl">

لعرض صورة من الدفعة: `images[i]` شكلها `(1, 28, 28)`، لذلك مرّر `images[i][0]` إلى `plt.imshow` مع `cmap='gray'`. ولقراءة الـ label كرقم عادي استخدم `labels[i].item()`.

</div>

</details>

<details>
<summary>💡 Hint 3</summary>

<div dir="rtl">

```python
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)

fc_model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 128), nn.ReLU(),
    # ... أكمل
)
```

</div>

</details>

### 🧑‍💻 Your Solution

In [ ]:
# TODO: أكمل التنفيذ

# Step 1: التطبيع
transform = None

# Step 2: تحميل مجموعتَي التدريب والاختبار
train_dataset = None
test_dataset  = None

# Step 3: الدفعات
train_loader = None
test_loader  = None

# Step 4: افحص أول دفعة واطبع الأبعاد ومجال القيم
images, labels = None, None

# Step 5: اعرض 10 صور في شبكة 2×5

# Step 6: ابنِ الشبكة العادية
fc_model = None

# Step 7: احسب عدد الأوزان
fc_params = None

# Write your solution below

### ✅ Self-Check

In [ ]:
assert train_dataset is not None and test_dataset is not None, "لم يتم تحميل مجموعات البيانات."
assert len(train_dataset) == 60000, f"عدد صور التدريب خاطئ: {len(train_dataset)} (المتوقع 60000)."
assert len(test_dataset)  == 10000, f"عدد صور الاختبار خاطئ: {len(test_dataset)} (المتوقع 10000)."

assert train_loader.batch_size == 64, f"حجم الدفعة خاطئ: {train_loader.batch_size} (المتوقع 64)."
assert tuple(images.shape) == (64, 1, 28, 28), \
    f"شكل الدفعة خاطئ: {tuple(images.shape)} (المتوقع (64, 1, 28, 28))."
assert tuple(labels.shape) == (64,), f"شكل الـ labels خاطئ: {tuple(labels.shape)}."
assert images.min() < -0.5 and images.max() > 0.5, \
    "قيم البكسلات لا تبدو مُطبَّعة — هل نسيت transforms.Normalize((0.5,), (0.5,))؟"

assert fc_model is not None, "الشبكة 'fc_model' لم تُبنَ."
assert isinstance(fc_model[0], nn.Flatten), "الطبقة الأولى يجب أن تكون nn.Flatten."
out = fc_model(images)
assert tuple(out.shape) == (64, 10), \
    f"خرج الشبكة خاطئ: {tuple(out.shape)} (المتوقع (64, 10) — عصبون لكل رقم)."
assert fc_params == 109386, \
    f"عدد الأوزان خاطئ: {fc_params} (المتوقع 109,386) — تحقق من أحجام الطبقات 784→128→64→10."

print("✅ Basic checks passed.")
print(f"   دفعة: {tuple(images.shape)} | مجال القيم: [{images.min():.2f}, {images.max():.2f}]")
print(f"   عدد أوزان الشبكة العادية: {fc_params:,}")

### 💡 Solution

<details>
<summary>💡 Show Solution</summary>

#### Approach

<div dir="rtl">

نبني خط أنابيب البيانات كاملاً: تحويل الصورة إلى tensor، تطبيعها، تقسيمها إلى دفعات. ثم نفحص الأبعاد قبل بناء الشبكة — لأن أول `Linear` يجب أن يستقبل 784 = 28×28 مدخلاً بالضبط.

</div>

#### Reference Implementation

```python
# --- تجهيز البيانات ---
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_dataset  = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_dataset,  batch_size=64, shuffle=False)

print(f"📚 صور التدريب: {len(train_dataset):,} | صور الاختبار: {len(test_dataset):,}")

# --- فحص أول دفعة ---
images, labels = next(iter(train_loader))
print("images.shape =", tuple(images.shape))
print("labels.shape =", tuple(labels.shape))
print(f"مجال القيم بعد التطبيع: [{images.min():.2f}, {images.max():.2f}]")

# --- عرض العينات ---
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle(ar("عينات من الأرقام"), fontsize=14)
for i, axi in enumerate(axes.flat):
    axi.imshow(images[i][0], cmap='gray')
    axi.set_title(f'{labels[i].item()}')
    axi.axis('off')
plt.show()

# --- بناء الشبكة العادية ---
fc_model = nn.Sequential(
    nn.Flatten(),           # 28×28 → 784  (نسطّح الصورة! 🚩)
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 10)       # مخرج: 10 أرقام
)

print(fc_model)
fc_params = sum(p.numel() for p in fc_model.parameters())
print(f"\n👥 عدد الأوزان: {fc_params:,}")
```

#### Explanation

<div dir="rtl">

- `ToTensor()` تحوّل الصورة إلى tensor بأبعاد `(C, H, W)` وقيم بين 0 و 1.
- `Normalize((0.5,), (0.5,))` تنقلها إلى المجال `[-1, 1]` — أرقام صغيرة متمركزة حول الصفر تُسهّل التعلم.
- `DataLoader` يقسم 60,000 صورة إلى دفعات من 64 (أي 938 دفعة تقريباً)، و `shuffle=True` يمنع الشبكة من حفظ ترتيب البيانات.
- عدد الأوزان: `(784×128 + 128) + (128×64 + 64) + (64×10 + 10) = 109,386`.

</div>

#### Expected Result

```
📚 صور التدريب: 60,000 | صور الاختبار: 10,000
images.shape = (64, 1, 28, 28)
labels.shape = (64,)
مجال القيم بعد التطبيع: [-1.00, 1.00]
👥 عدد الأوزان: 109,386
```

#### Common Mistakes

<div dir="rtl">

- كتابة `Normalize(0.5, 0.5)` بدون أقواس tuple — تُنتج خطأ في بعض الإصدارات.
- استخدام `shuffle=True` لمجموعة الاختبار — يجعل مقارنة النتائج بين التشغيلات أصعب.
- نسيان `nn.Flatten()` كطبقة أولى فتفشل `nn.Linear` لأن المدخل رباعي الأبعاد.
- كتابة `nn.Linear(28, 128)` بدل `nn.Linear(784, 128)`.

</div>

</details>

---

<div dir="rtl">
<h2>Exercise 7 — حلقة التدريب بخطواتها الأربع | The 4-Step Training Loop</h2>

<h3 style="direction: rtl;">🎯 Learning Objective</h3>
<p style="direction: rtl; text-align: right;">أن تكتب <strong>حلقة التدريب (Training Loop)</strong> كاملة بخطواتها الأربع الثابتة، وتدرّب الشبكة على MNIST، وترسم <strong>منحنى التعلّم</strong>، وتقيّم الدقة على بيانات لم ترها الشبكة من قبل.</p>

<h3 style="direction: rtl;">🧩 Concept</h3>
<p style="direction: rtl; text-align: right;"><strong>حلقة التدريب</strong> = <code>zero_grad()</code> → <strong>Forward Pass</strong> → <code>backward()</code> → <code>step()</code>، مع <strong>Cross-Entropy Loss</strong> و <strong>Adam Optimizer</strong> و <strong>Epochs</strong> والتقييم على مجموعة الاختبار.</p>

<h3 style="direction: rtl;">📖 Scenario</h3>
<p style="direction: rtl; text-align: right;">هذه الخطوات الأربع هي <strong>نفسها في كل مشروع PyTorch ستكتبه في حياتك</strong> — سواء كنت تدرّب شبكة صغيرة على MNIST أو نموذجاً ضخماً على ملايين الصور. احفظها الآن بيدك، لا بعينك.</p>

<h3 style="direction: rtl;">📝 Task</h3>
<ol style="direction: rtl; text-align: right;">
<li>عرّف <code>loss_fn = nn.CrossEntropyLoss()</code> و <code>optimizer = torch.optim.Adam(fc_model.parameters(), lr=0.001)</code>.</li>
<li>اكتب حلقة تدريب لـ <code>epochs = 3</code> تمر على <code>train_loader</code>، وتُنفّذ في كل دفعة <strong>الخطوات الأربع بالترتيب</strong>:
   <ol style="direction: rtl; text-align: right; list-style-type: decimal; margin-top: 5px;">
   <li><code>optimizer.zero_grad()</code> — تصفير التدرجات</li>
   <li><code>outputs = fc_model(images)</code> — التمرير الأمامي ثم <code>loss = loss_fn(outputs, labels)</code></li>
   <li><code>loss.backward()</code> — الانتشار الخلفي</li>
   <li><code>optimizer.step()</code> — تحديث الأوزان</li>
   </ol>
</li>
<li>في كل دورة، احسب <strong>متوسط الخسارة</strong> و <strong>الدقة</strong> على بيانات التدريب، وخزّنهما في <code>fc_loss_history</code> و <code>fc_accuracy_history</code>.</li>
<li>ارسم منحنى الخسارة ومنحنى الدقة جنباً إلى جنب باستخدام <code>plt.subplots(1, 2, figsize=(13, 4))</code>.</li>
<li>قيّم النموذج على <code>test_loader</code> داخل <code>with torch.no_grad():</code> واحسب <code>fc_accuracy</code> (نسبة بين 0 و 1).</li>
<li>اعرض توقعات الشبكة على 10 صور اختبار مع تلوين العنوان بالأخضر عند الإصابة والأحمر عند الخطأ.</li>
</ol>

<h3 style="direction: rtl;">Requirements</h3>
<ul style="direction: rtl; text-align: right;">
<li><strong>المكتبات:</strong> <code>torch</code>, <code>torch.nn as nn</code>, <code>matplotlib.pyplot</code>, <code>tqdm</code> (اختياري لشريط التقدّم).</li>
<li><strong>المتغيرات المطلوبة:</strong> <code>loss_fn</code>, <code>optimizer</code>, <code>fc_loss_history</code>, <code>fc_accuracy_history</code>, <code>fc_accuracy</code>.</li>
<li><strong>الدوال المستخدمة:</strong> <code>torch.max(outputs, 1)</code> لاستخراج التوقع، و <code>(predicted == labels).sum().item()</code> لعدّ الإصابات.</li>
<li><strong>قيود:</strong> لا تستخدم أي دالة تدريب جاهزة من مكتبة خارجية — اكتب الحلقة بنفسك. وأثناء التقييم يجب استخدام <code>torch.no_grad()</code>.</li>
</ul>

<h3 style="direction: rtl;">Expected Result</h3>
<ul style="direction: rtl; text-align: right;">
<li>الخسارة يجب أن <strong>تنخفض</strong> في كل دورة والدقة أن <strong>ترتفع</strong> — هذا هو الدليل البصري على أن الشبكة تتعلم فعلاً. بعد 3 دورات ستحصل على دقة اختبار <strong>تتجاوز 95%</strong> بوضوح. وفي عرض التوقعات، ستجد أغلب العناوين خضراء وواحداً أو اثنين حمراء على صور مشوّهة يصعب تمييزها حتى على البشر.</li>
</ul>

</div>

### 💡 Hints

<details>
<summary>💡 Hint 1</summary>

<div dir="rtl">

الترتيب مهم جداً: التصفير **قبل** التمرير الأمامي، و `backward()` **قبل** `step()`. لو نسيت `zero_grad()` ستتراكم تدرجات الدفعات فوق بعضها (تذكّر تجميع التدرجات عند التفرّع في التمرين الثالث!).

</div>

</details>

<details>
<summary>💡 Hint 2</summary>

<div dir="rtl">

لحساب الدقة: `_, predicted = torch.max(outputs, 1)` تُعيد فهرس أعلى قيمة في كل صف — أي الرقم الذي اختارته الشبكة. ثم `(predicted == labels).sum().item()` يعطيك عدد الإصابات في الدفعة.

</div>

</details>

<details>
<summary>💡 Hint 3</summary>

<div dir="rtl">

```python
for epoch in range(epochs):
    total_correct, total_samples, epoch_loss = 0, 0, 0.0
    for images, labels in tqdm(train_loader, desc=f'دورة {epoch+1}/{epochs}'):
        optimizer.zero_grad()             # 1
        outputs = fc_model(images)        # 2
        loss = loss_fn(outputs, labels)
        loss.backward()                   # 3
        optimizer.step()                  # 4
        epoch_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total_correct += (predicted == labels).sum().item()
        total_samples += labels.size(0)
    fc_loss_history.append(epoch_loss / len(train_loader))
    fc_accuracy_history.append(total_correct / total_samples)
```

</div>

</details>

### 🧑‍💻 Your Solution

In [ ]:
# TODO: أكمل التنفيذ

# Step 1: دالة الخسارة والمُحسّن
loss_fn   = None
optimizer = None

epochs = 3
fc_loss_history = []
fc_accuracy_history = []

# Step 2: حلقة التدريب بالخطوات الأربع
# for epoch in range(epochs):
#     ...

# Step 3: ارسم منحنى الخسارة ومنحنى الدقة

# Step 4: التقييم على مجموعة الاختبار
fc_accuracy = None

# Step 5: اعرض توقعات الشبكة على 10 صور اختبار
# Write your solution below

### ✅ Self-Check

In [ ]:
assert isinstance(loss_fn, nn.CrossEntropyLoss), \
    "دالة الخسارة يجب أن تكون nn.CrossEntropyLoss — المسألة تصنيف وليست انحدار."
assert isinstance(optimizer, torch.optim.Adam), "المُحسّن المطلوب هو Adam."

assert len(fc_loss_history) == epochs, f"طول سجل الخسارة خاطئ: {len(fc_loss_history)}."
assert len(fc_accuracy_history) == epochs, f"طول سجل الدقة خاطئ: {len(fc_accuracy_history)}."

assert fc_loss_history[-1] < fc_loss_history[0], \
    "الخسارة لم تنخفض! تحقق من ترتيب الخطوات الأربع في الحلقة."
assert fc_accuracy_history[-1] > fc_accuracy_history[0], "الدقة لم ترتفع خلال التدريب."

assert fc_accuracy is not None, "المتغير 'fc_accuracy' لم يُحسب."
assert 0.0 <= fc_accuracy <= 1.0, f"الدقة يجب أن تكون نسبة بين 0 و 1، لكنها {fc_accuracy}."
assert fc_accuracy > 0.95, f"الدقة على مجموعة الاختبار منخفضة: {fc_accuracy:.2%} (المتوقع > 95%)."

# التحقق من أن الأوزان تغيّرت فعلاً (أي أن optimizer.step() نُفّذ)
assert any(p.grad is not None for p in fc_model.parameters()), \
    "لا توجد تدرجات محفوظة — هل نفّذت loss.backward()؟"

print("✅ Basic checks passed.")
print(f"   الخسارة: {fc_loss_history[0]:.4f} → {fc_loss_history[-1]:.4f}")
print(f"   دقة الاختبار: {fc_accuracy:.2%}")

### 💡 Solution

<details>
<summary>💡 Show Solution</summary>

#### Approach

<div dir="rtl">

نُكرّر الخطوات الأربع على كل دفعة من `train_loader`، ونجمع الإحصاءات لرسم منحنى التعلّم. ثم ننتقل إلى وضع التقييم ونمرّر بيانات الاختبار **بدون حساب تدرجات** لأننا لا نُحدّث أوزاناً.

</div>

#### Reference Implementation

```python
loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(fc_model.parameters(), lr=0.001)

epochs = 3
fc_loss_history, fc_accuracy_history = [], []

for epoch in range(epochs):
    total_correct, total_samples, epoch_loss = 0, 0, 0.0
    progress = tqdm(train_loader, desc=f'دورة {epoch+1}/{epochs}')

    for images, labels in progress:
        optimizer.zero_grad()             # 1) صفّر التدرجات
        outputs = fc_model(images)        # 2) التمرير الأمامي
        loss = loss_fn(outputs, labels)   #    قِس الخسارة
        loss.backward()                   # 3) الانتشار الخلفي
        optimizer.step()                  # 4) المُحسّن يعدّل الأوزان

        epoch_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total_correct += (predicted == labels).sum().item()
        total_samples += labels.size(0)
        progress.set_postfix({'الدقة': f'{total_correct/total_samples:.3f}'})

    fc_loss_history.append(epoch_loss / len(train_loader))
    fc_accuracy_history.append(total_correct / total_samples)
    print(f"الدورة {epoch+1}: دقة {fc_accuracy_history[-1]:.2%}")

# --- منحنى التعلّم ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(range(1, epochs+1), fc_loss_history, 'o-', color='#dc2626', lw=2)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title(ar('الخسارة تنخفض')); ax1.grid(alpha=0.3)
ax2.plot(range(1, epochs+1), [a*100 for a in fc_accuracy_history], 'o-', color='#16a34a', lw=2)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)'); ax2.set_title(ar('الدقة ترتفع')); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# --- التقييم ---
fc_model.eval()
correct, total = 0, 0
with torch.no_grad():
    for images_t, labels_t in tqdm(test_loader, desc='جاري الاختبار'):
        outputs = fc_model(images_t)
        _, predicted = torch.max(outputs, 1)
        total   += labels_t.size(0)
        correct += (predicted == labels_t).sum().item()

fc_accuracy = correct / total
print(f"\n🎯 الدقة النهائية: {fc_accuracy:.2%} ({correct:,} من {total:,})")

# --- عرض التوقعات ---
test_images, test_labels = next(iter(test_loader))
with torch.no_grad():
    _, predicted_labels = torch.max(fc_model(test_images), 1)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle(ar("توقعات الشبكة"), fontsize=14)
for i, axi in enumerate(axes.flat):
    axi.imshow(test_images[i][0], cmap='gray')
    pred, true = predicted_labels[i].item(), test_labels[i].item()
    axi.set_title(ar(f'توقع: {pred}\nحقيقة: {true}'), color=('green' if pred == true else 'red'))
    axi.axis('off')
plt.show()

fc_model.train()
```

#### Explanation

<div dir="rtl">

- `zero_grad()` ضرورية لأن PyTorch **يُراكم** التدرجات — بدونها تختلط تدرجات الدفعة الحالية بالسابقة.
- `loss.backward()` هو ما درسناه في التمرينين 2 و 3، لكن على 109,386 وزناً بدل أربعة مدخلات.
- `optimizer.step()` يطبّق قاعدة التحديث من التمرين 5 على كل وزن.
- `torch.no_grad()` أثناء التقييم توفّر الذاكرة والوقت لأننا لا نحتاج بناء رسم الاشتقاق.

</div>

#### Expected Result

```
الدورة 1: دقة 92.xx%
الدورة 2: دقة 96.xx%
الدورة 3: دقة 97.xx%
🎯 الدقة النهائية: ~97%
```

<div dir="rtl">

(الأرقام تتغيّر قليلاً بين التشغيلات حسب البذرة العشوائية.)

</div>

#### Common Mistakes

<div dir="rtl">

- وضع `optimizer.zero_grad()` بعد `loss.backward()` — فتُمسح التدرجات قبل استخدامها ولا تتعلم الشبكة شيئاً.
- استخدام `loss` بدل `loss.item()` عند التجميع — يُبقي رسم الاشتقاق في الذاكرة ويستهلكها كلها.
- نسيان `torch.no_grad()` أثناء التقييم — يعمل الكود لكنه أبطأ ويستهلك ذاكرة بلا داعٍ.
- حساب الدقة على بيانات التدريب فقط والاعتقاد أن الشبكة "تعمّم" — القياس الحقيقي على `test_loader`.

</div>

</details>

---

<div dir="rtl">
<h2>Exercise 8 — إثبات مشكلة التسطيح | Proving The Flattening Problem</h2>

<h3 style="direction: rtl;">🎯 Learning Objective</h3>
<p style="direction: rtl; text-align: right;">أن تُثبت <strong>تجريبياً</strong> أن الشبكة العادية لا ترى البنية المكانية للصورة إطلاقاً: ستخلط ترتيب البكسلات خلطاً عشوائياً ثابتاً، وتُعيد التدريب، وتكتشف أن الدقة <strong>لم تتغيّر تقريباً</strong>.</p>

<h3 style="direction: rtl;">🧩 Concept</h3>
<p style="direction: rtl; text-align: right;"><strong>مشكلة تسطيح الصورة (The Flattening Problem)</strong>: <code>nn.Flatten()</code> تُدمّر <strong>البنية المكانية (Spatial Structure)</strong> — بكسلان متجاوران رأسياً يصبحان بعيدين في المصفوفة الأحادية، والشبكة <strong>لا تعرف</strong> أنهما كانا جارين.</p>

<div align="center">
<img src="media/01-flatting-problem.gif" width="620">
</div>

<h3 style="direction: rtl;">📖 Scenario</h3>
<p style="direction: rtl; text-align: right;">حققنا دقة ~97% في التمرين السابق، لكن أول سطر في الشبكة كان <code>nn.Flatten()</code>. هل هذا فعلاً مشكلة؟ الاختبار الحاسم: لو خلطنا مواضع البكسلات بترتيب عشوائي <strong>ثابت</strong> (نفس الخلطة لكل الصور)، فالصورة تصبح غير مفهومة تماماً لعين الإنسان. فإذا بقيت دقة الشبكة كما هي، فهذا <strong>برهان قاطع</strong> على أنها لم تكن تستفيد من البنية المكانية أصلاً.</p>

<h3 style="direction: rtl;">📝 Task</h3>
<ol style="direction: rtl; text-align: right;">
<li>أنشئ ترتيباً عشوائياً ثابتاً للبكسلات: <code>perm = torch.randperm(784)</code> (استخدم <code>torch.manual_seed(0)</code> قبلها لتثبيت النتيجة).</li>
<li>اكتب دالة <code>shuffle_pixels(batch, perm)</code> تُسطّح الدفعة إلى <code>(B, 784)</code>، تُعيد ترتيب الأعمدة حسب <code>perm</code>، ثم تُعيد تشكيلها إلى <code>(B, 1, 28, 28)</code>.</li>
<li>اعرض مقارنة بصرية: صورة أصلية بجانب نسختها المخلوطة (<code>plt.subplots(1, 2)</code>).</li>
<li>ابنِ شبكة جديدة <code>shuffled_model</code> بنفس بنية <code>fc_model</code> بالضبط.</li>
<li>درّبها <strong>دورة واحدة (epoch)</strong> على البيانات المخلوطة: طبّق <code>shuffle_pixels</code> على كل دفعة قبل تمريرها للنموذج.</li>
<li>قيّمها على بيانات الاختبار <strong>المخلوطة بنفس <code>perm</code></strong> واحسب <code>shuffled_accuracy</code>.</li>
<li>اطبع مقارنة بين <code>shuffled_accuracy</code> ودقة شبكة عادية دُرّبت دورة واحدة على البيانات الأصلية (استخدم <code>fc_accuracy_history[0]</code> أو درّب نموذجاً مرجعياً دورة واحدة).</li>
</ol>

<h3 style="direction: rtl;">Requirements</h3>
<ul style="direction: rtl; text-align: right;">
<li><strong>المكتبات:</strong> <code>torch</code>, <code>torch.nn as nn</code>, <code>matplotlib.pyplot</code>.</li>
<li><strong>المتغيرات المطلوبة:</strong> <code>perm</code>, <code>shuffle_pixels</code>, <code>shuffled_model</code>, <code>shuffled_accuracy</code>.</li>
<li><strong>الأبعاد:</strong> <code>perm.shape == (784,)</code>، ومخرج <code>shuffle_pixels</code> بنفس شكل المدخل <code>(B, 1, 28, 28)</code>.</li>
<li><strong>قيود:</strong> يجب استخدام <strong>نفس</strong> <code>perm</code> في التدريب والاختبار (وإلا فالخلط عشوائي لكل دفعة وستنهار الدقة لسبب مختلف تماماً). استخدم نفس بنية الشبكة ونفس المُحسّن لتكون المقارنة عادلة.</li>
</ul>

<h3 style="direction: rtl;">Expected Result</h3>
<ul style="direction: rtl; text-align: right;">
<li>الصورة المخلوطة ستبدو <strong>ضجيجاً كاملاً</strong> لا يمكن لأي إنسان التعرف على الرقم فيها. ورغم ذلك، دقة الشبكة على البيانات المخلوطة ستكون <strong>قريبة جداً</strong> من دقتها على البيانات الأصلية (الفرق عادةً أقل من بضع نقاط مئوية). الاستنتاج: الشبكة العادية تعامل الصورة كـ <strong>قائمة أرقام بلا ترتيب</strong> — البنية المكانية ضاعت في <code>nn.Flatten()</code>. وهذا بالضبط ما ستحلّه الـ Convolution في التمرين القادم.</li>
</ul>

</div>

### 💡 Hints

<details>
<summary>💡 Hint 1</summary>

<div dir="rtl">

لإعادة ترتيب الأعمدة في tensor مسطّح: `flat[:, perm]` حيث `flat` شكلها `(B, 784)` و `perm` متجه فهارس. هذه العملية تُبقي **نفس مجموعة القيم** لكن في مواضع مختلفة.

</div>

</details>

<details>
<summary>💡 Hint 2</summary>

<div dir="rtl">

```python
def shuffle_pixels(batch, perm):
    B = batch.shape[0]
    flat = batch.view(B, -1)      # (B, 784)
    return flat[:, perm].view(B, 1, 28, 28)
```

طبّقها داخل حلقة التدريب مباشرة: `images = shuffle_pixels(images, perm)` قبل `shuffled_model(images)`.

</div>

</details>

<details>
<summary>💡 Hint 3</summary>

<div dir="rtl">

حلقة التدريب هي نفسها من التمرين السابع حرفياً، مع إضافة سطر الخلط:

```python
for images, labels in train_loader:
    images = shuffle_pixels(images, perm)   # ← السطر الوحيد الجديد
    optimizer2.zero_grad()
    outputs = shuffled_model(images)
    ...
```

</div>

</details>

### 🧑‍💻 Your Solution

In [ ]:
# TODO: أكمل التنفيذ

# Step 1: ترتيب عشوائي ثابت للبكسلات
torch.manual_seed(0)
perm = None

# Step 2: دالة الخلط
def shuffle_pixels(batch, perm):
    pass

# Step 3: قارن بصرياً بين صورة أصلية ونسختها المخلوطة

# Step 4: شبكة جديدة بنفس البنية
shuffled_model = None

# Step 5: درّبها دورة واحدة على البيانات المخلوطة

# Step 6: قيّمها على بيانات الاختبار المخلوطة
shuffled_accuracy = None

# Step 7: اطبع المقارنة مع الشبكة العادية
# Write your solution below

### ✅ Self-Check

In [ ]:
assert perm is not None and tuple(perm.shape) == (784,), \
    f"شكل perm خاطئ: {tuple(perm.shape) if perm is not None else None} (المتوقع (784,))."
assert len(torch.unique(perm)) == 784, "perm يجب أن يحتوي كل فهرس مرة واحدة — استخدم torch.randperm."

sample = images[:4]
shuffled_sample = shuffle_pixels(sample, perm)
assert tuple(shuffled_sample.shape) == tuple(sample.shape), \
    f"شكل المخرج خاطئ: {tuple(shuffled_sample.shape)} — يجب أن يبقى (B, 1, 28, 28)."
assert torch.allclose(shuffled_sample.view(4, -1).sort(dim=1).values,
                      sample.view(4, -1).sort(dim=1).values), \
    "الخلط غيّر قيم البكسلات! يجب أن تبقى نفس القيم بمواضع مختلفة فقط."
assert not torch.allclose(shuffled_sample, sample), "الصورة لم تتغيّر — هل طبّقت perm فعلاً؟"

assert shuffled_model is not None, "الشبكة 'shuffled_model' لم تُبنَ."
assert shuffled_accuracy is not None, "المتغير 'shuffled_accuracy' لم يُحسب."
assert shuffled_accuracy > 0.85, (
    f"الدقة على البيانات المخلوطة منخفضة جداً ({shuffled_accuracy:.2%}). "
    "تأكد أنك تستخدم نفس perm في التدريب والاختبار.")

gap = abs(shuffled_accuracy - fc_accuracy_history[0])
assert gap < 0.06, (
    f"الفارق كبير ({gap:.2%}) — قارن مع دقة الدورة الأولى للشبكة العادية بنفس الشروط.")

print("✅ Basic checks passed.")
print(f"   دقة الشبكة العادية (دورة 1): {fc_accuracy_history[0]:.2%}")
print(f"   دقة الشبكة على بكسلات مخلوطة: {shuffled_accuracy:.2%}")
print("   👉 الشبكة العادية لا ترى البنية المكانية إطلاقاً!")

### 💡 Solution

<details>
<summary>💡 Show Solution</summary>

#### Approach

<div dir="rtl">

نُطبّق تبديلاً ثابتاً على مواضع البكسلات. الصورة تفقد كل معناها البصري، لكن **مجموعة القيم** تبقى كما هي. بما أن الشبكة العادية تُسطّح الصورة فوراً، فكل ما تراه هو 784 رقماً بترتيب ثابت — ولا يهمها أي ترتيب هو، ما دام **ثابتاً** عبر كل الصور.

</div>

#### Reference Implementation

```python
torch.manual_seed(0)
perm = torch.randperm(784)

def shuffle_pixels(batch, perm):
    B = batch.shape[0]
    flat = batch.view(B, -1)                  # (B, 784)
    return flat[:, perm].view(B, 1, 28, 28)   # نفس القيم، مواضع مختلفة

# --- المقارنة البصرية ---
sample_img = images[0:1]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
ax1.imshow(sample_img[0][0], cmap='gray'); ax1.set_title(ar('الصورة الأصلية')); ax1.axis('off')
ax2.imshow(shuffle_pixels(sample_img, perm)[0][0], cmap='gray')
ax2.set_title(ar('بعد خلط البكسلات')); ax2.axis('off')
plt.show()

# --- شبكة جديدة بنفس البنية ---
torch.manual_seed(42)
shuffled_model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 128), nn.ReLU(),
    nn.Linear(128, 64),  nn.ReLU(),
    nn.Linear(64, 10)
)

loss_fn2   = nn.CrossEntropyLoss()
optimizer2 = torch.optim.Adam(shuffled_model.parameters(), lr=0.001)

# --- تدريب دورة واحدة على البيانات المخلوطة ---
for imgs, lbls in tqdm(train_loader, desc='تدريب على بكسلات مخلوطة'):
    imgs = shuffle_pixels(imgs, perm)         # ← السطر الوحيد الجديد
    optimizer2.zero_grad()
    outputs = shuffled_model(imgs)
    loss = loss_fn2(outputs, lbls)
    loss.backward()
    optimizer2.step()

# --- التقييم على اختبار مخلوط بنفس perm ---
shuffled_model.eval()
correct, total = 0, 0
with torch.no_grad():
    for imgs, lbls in test_loader:
        imgs = shuffle_pixels(imgs, perm)
        _, predicted = torch.max(shuffled_model(imgs), 1)
        total   += lbls.size(0)
        correct += (predicted == lbls).sum().item()

shuffled_accuracy = correct / total

print(f"دقة الشبكة العادية بعد دورة واحدة : {fc_accuracy_history[0]:.2%}")
print(f"دقة الشبكة على بكسلات مخلوطة      : {shuffled_accuracy:.2%}")
print("👉 الفرق ضئيل — الشبكة لم تكن تستخدم البنية المكانية أصلاً!")
```

#### Explanation

<div dir="rtl">

- التبديل الثابت يُكافئ إعادة ترتيب أعمدة مصفوفة الأوزان في الطبقة الأولى — والشبكة تتعلم الترتيب الجديد بنفس السهولة تماماً.
- بالنسبة للعين البشرية الصورة صارت ضجيجاً، لكن بالنسبة لـ `nn.Linear` لا شيء تغيّر: هي أصلاً لا تعرف أن البكسل رقم 5 والبكسل رقم 33 كانا فوق بعضهما.
- **الخلاصة:** المعلومات المحلية (Local Context) تضيع تماماً في التسطيح. نحتاج عملية تحترم الجيرة المكانية — وهي **Convolution**.

</div>

#### Expected Result

```
دقة الشبكة العادية بعد دورة واحدة : ~92%
دقة الشبكة على بكسلات مخلوطة      : ~92%
👉 الفرق ضئيل — الشبكة لم تكن تستخدم البنية المكانية أصلاً!
```

#### Common Mistakes

<div dir="rtl">

- توليد `perm` جديد داخل الحلقة — فيصبح كل دفعة مخلوطة بشكل مختلف، وتنهار الدقة لسبب لا علاقة له بالتسطيح.
- نسيان تطبيق الخلط على بيانات **الاختبار** أيضاً.
- تدريب `shuffled_model` عدداً مختلفاً من الدورات ثم مقارنته بشبكة دُرّبت 3 دورات — مقارنة غير عادلة.
- استخدام `.reshape` على tensor غير متجاور دون انتباه؛ `view` بعد `[:, perm]` تعمل هنا لأن الفهرسة المتقدّمة تُنتج نسخة متجاورة.

</div>

</details>

---

<div dir="rtl">
<h2>Exercise 9 — الالتفاف ثنائي البعد من الصفر | 2D Convolution from Scratch</h2>

<h3 style="direction: rtl;">🎯 Learning Objective</h3>
<p style="direction: rtl; text-align: right;">أن تنفّذ <strong>عملية الالتفاف (Convolution)</strong> بنفسك بحلقات صريحة، وتتحقق من <strong>معادلة حجم المخرج</strong>، وتطبّق <strong>Kernels متعددة</strong> على صورة MNIST حقيقية لتُنتج <strong>خرائط ميزات (Feature Maps)</strong> مختلفة.</p>

<h3 style="direction: rtl;">🧩 Concept</h3>
<p style="direction: rtl; text-align: right;"><strong>الالتفاف (Convolution)</strong>، <strong>نواة الالتفاف (Kernel / Filter)</strong>، <strong>خريطة الميزات (Feature Map)</strong>، <strong>الـ Kernels المتعددة والقنوات (Channels)</strong>:</p>
<p style="direction: rtl; text-align: center; font-size: 1.1em;">$$H' = H - K + 1 \qquad W' = W - K + 1 \qquad C' = \text{عدد الـ Kernels}$$</p>

<h3 style="direction: rtl;">📖 Scenario</h3>
<p style="direction: rtl; text-align: right;">في التمرين السابق أثبتنا أن التسطيح يُدمّر البنية المكانية. الحل: عملية تنظر إلى <strong>نافذة صغيرة</strong> من الصورة في كل مرة — تضرب قيم النافذة في قيم الـ Kernel وتجمعها لتُنتج رقماً واحداً. هذه هي العملية التي تقوم عليها رؤية الحاسوب الحديثة كلها، وستكتبها الآن بيدك في عشرة أسطر.</p>

<div align="center">
<img src="media/02-convolution-operation.png" width="640">
</div>

<h3 style="direction: rtl;">📝 Task</h3>
<ol style="direction: rtl; text-align: right;">
<li>اكتب دالة <code>my_conv2d(image, kernel)</code>:
   <ul style="direction: rtl; text-align: right; margin-top: 5px;">
   <li><code>image</code> بشكل <code>(H, W)</code> و <code>kernel</code> بشكل <code>(K, K)</code> — كلاهما <code>torch.Tensor</code>.</li>
   <li>احسب <code>H' = H - K + 1</code> و <code>W' = W - K + 1</code> وأنشئ tensor للمخرج بهذا الحجم.</li>
   <li>لكل موضع <code>(i, j)</code>: اقتطع النافذة <code>image[i:i+K, j:j+K]</code>، اضربها عنصراً بعنصر في الـ kernel، واجمع الناتج.</li>
   <li>أعِد خريطة الميزات.</li>
   </ul>
</li>
<li>اكتب دالة <code>conv_output_shape(H, W, K)</code> تُعيد <code>(H', W')</code> حسب المعادلة.</li>
<li>اختبر دالتك على مصفوفة <code>6×6</code> مع kernel <code>3×3</code> وتأكد أن المخرج <code>4×4</code>.</li>
<li>تحقّق من صحة تنفيذك بمقارنته مع <code>torch.nn.functional.conv2d</code> (لاحظ أنها تحتاج أبعاداً بالشكل <code>(1, 1, H, W)</code> و <code>(1, 1, K, K)</code>).</li>
<li>طبّق <strong>ثلاث Kernels مختلفة</strong> على صورة رقم واحدة من MNIST:
   <ul style="direction: rtl; text-align: right; margin-top: 5px;">
   <li><code>vertical_kernel</code> = <code>[[1,0,-1],[1,0,-1],[1,0,-1]]</code> (حواف رأسية)</li>
   <li><code>horizontal_kernel</code> = منقول الأولى (حواف أفقية)</li>
   <li><code>blur_kernel</code> = <code>torch.ones(3,3) / 9</code> (تنعيم)</li>
   </ul>
</li>
<li>كدّس الخرائط الثلاث في tensor واحد اسمه <code>feature_maps</code> بشكل <code>(3, 26, 26)</code>، واعرضها بجانب الصورة الأصلية.</li>
</ol>

<h3 style="direction: rtl;">Requirements</h3>
<ul style="direction: rtl; text-align: right;">
<li><strong>المكتبات:</strong> <code>torch</code>, <code>torch.nn.functional as F</code>, <code>matplotlib.pyplot</code>.</li>
<li><strong>الدوال المطلوبة:</strong> <code>my_conv2d(image, kernel)</code> و <code>conv_output_shape(H, W, K)</code>.</li>
<li><strong>المتغيرات المطلوبة:</strong> <code>vertical_kernel</code>, <code>horizontal_kernel</code>, <code>blur_kernel</code>, <code>feature_maps</code>, <code>digit</code>.</li>
<li><strong>الأبعاد:</strong> الصورة <code>digit</code> بشكل <code>(28, 28)</code>، والـ kernels بشكل <code>(3, 3)</code>، و <code>feature_maps</code> بشكل <code>(3, 26, 26)</code>.</li>
<li><strong>قيود:</strong> في <code>my_conv2d</code> ممنوع استخدام <code>F.conv2d</code> أو <code>nn.Conv2d</code> — يجب أن تكتب الحلقات بنفسك. بدون padding وبخطوة (stride) = 1.</li>
</ul>

<h3 style="direction: rtl;">Expected Result</h3>
<ul style="direction: rtl; text-align: right;">
<li>خرجك يجب أن يطابق <code>F.conv2d</code> تماماً. أما بصرياً فستلاحظ اختلافاً واضحاً بين الخرائط الثلاث: خريطة الحواف الرأسية تُضيء عند الحدود <strong>العمودية</strong> للرقم، وخريطة الحواف الأفقية تُضيء عند الحدود <strong>الأفقية</strong>، بينما خريطة التنعيم تُنتج نسخة ضبابية من الرقم. <strong>نفس الصورة، ثلاث رؤى مختلفة</strong> — كل Kernel تكتشف نمطاً واحداً. ولاحظ أن الأبعاد تقلّصت من 28 إلى 26 لأننا لم نستخدم padding.</li>
</ul>

</div>

### 💡 Hints

<details>
<summary>💡 Hint 1</summary>

<div dir="rtl">

الخطوة الأساسية سطر واحد: `out[i, j] = (image[i:i+K, j:j+K] * kernel).sum()`. الضرب هنا **عنصر بعنصر (element-wise)** وليس ضرب مصفوفات — لا تستخدم `@` أو `torch.matmul`.

</div>

</details>

<details>
<summary>💡 Hint 2</summary>

<div dir="rtl">

لاستخراج صورة رقم واحدة من MNIST بشكل `(28, 28)`:

```python
digit, digit_label = test_dataset[0]   # digit.shape = (1, 28, 28)
digit = digit[0]                       # → (28, 28)
```

</div>

</details>

<details>
<summary>💡 Hint 3</summary>

<div dir="rtl">

```python
def my_conv2d(image, kernel):
    H, W = image.shape
    K = kernel.shape[0]
    Hp, Wp = H - K + 1, W - K + 1
    out = torch.zeros(Hp, Wp)
    for i in range(Hp):
        for j in range(Wp):
            out[i, j] = (image[i:i+K, j:j+K] * kernel).sum()
    return out

# للتحقق:
ref = F.conv2d(image.view(1,1,H,W), kernel.view(1,1,K,K)).view(Hp, Wp)
```

</div>

</details>

### 🧑‍💻 Your Solution

In [ ]:
# TODO: أكمل التنفيذ

# Step 1: نفّذ الالتفاف بالحلقات
def my_conv2d(image, kernel):
    # H, W = ...
    # K = ...
    # Hp, Wp = H - K + 1, W - K + 1
    # out = torch.zeros(Hp, Wp)
    # for i ... for j ...:  اقتطع النافذة، اضرب، اجمع
    pass

# Step 2: معادلة حجم المخرج
def conv_output_shape(H, W, K):
    pass

# Step 3: اختبار سريع على مصفوفة 6×6
test_image  = torch.arange(36, dtype=torch.float32).reshape(6, 6)
test_kernel = torch.tensor([[1., 0., -1.], [1., 0., -1.], [1., 0., -1.]])

# Step 4: تحقّق من المطابقة مع F.conv2d

# Step 5: عرّف الـ Kernels الثلاث وطبّقها على رقم من MNIST
digit = None
vertical_kernel   = None
horizontal_kernel = None
blur_kernel       = None

# Step 6: كدّس الخرائط واعرضها
feature_maps = None

# Write your solution below

### ✅ Self-Check

In [ ]:
assert 'my_conv2d' in dir(), "الدالة 'my_conv2d' لم تُعرَّف."
assert 'conv_output_shape' in dir(), "الدالة 'conv_output_shape' لم تُعرَّف."

assert conv_output_shape(6, 6, 3) == (4, 4), \
    f"معادلة الحجم خاطئة: conv_output_shape(6,6,3) = {conv_output_shape(6,6,3)} (المتوقع (4,4))."
assert conv_output_shape(224, 224, 3) == (222, 222), "المعادلة لا تعمل على أبعاد أكبر."

# مطابقة النتيجة مع PyTorch
_mine = my_conv2d(test_image, test_kernel)
_ref  = F.conv2d(test_image.view(1, 1, 6, 6), test_kernel.view(1, 1, 3, 3)).view(4, 4)
assert tuple(_mine.shape) == (4, 4), f"شكل المخرج خاطئ: {tuple(_mine.shape)} (المتوقع (4,4))."
assert torch.allclose(_mine, _ref, atol=1e-5), \
    "نتيجتك لا تطابق F.conv2d — تأكد أنك تضرب عنصراً بعنصر ثم تجمع، وأن النافذة تبدأ من [i:i+K, j:j+K]."

assert digit is not None and tuple(digit.shape) == (28, 28), \
    f"شكل 'digit' خاطئ: {tuple(digit.shape) if digit is not None else None} (المتوقع (28, 28))."

for name, k in [('vertical_kernel', vertical_kernel),
                ('horizontal_kernel', horizontal_kernel),
                ('blur_kernel', blur_kernel)]:
    assert k is not None and tuple(k.shape) == (3, 3), f"شكل '{name}' يجب أن يكون (3, 3)."

assert feature_maps is not None and tuple(feature_maps.shape) == (3, 26, 26), (
    f"شكل feature_maps خاطئ: {tuple(feature_maps.shape) if feature_maps is not None else None}. "
    "المتوقع (3, 26, 26): ثلاث Kernels ⇒ ثلاث قنوات خرج، و 28-3+1 = 26.")

assert not torch.allclose(feature_maps[0], feature_maps[1], atol=1e-4), \
    "خريطتا الحواف الرأسية والأفقية متطابقتان — تأكد أن الـ kernel الثانية هي منقول الأولى."
assert abs(blur_kernel.sum().item() - 1.0) < 1e-6, "مجموع kernel التنعيم يجب أن يساوي 1."

print("✅ Basic checks passed.")
print(f"   {tuple(digit.shape)} ⊛ (3,3) ⇒ {tuple(feature_maps.shape)}")

### 💡 Solution

<details>
<summary>💡 Show Solution</summary>

#### Approach

<div dir="rtl">

نُمرّر نافذة `K×K` فوق الصورة موضعاً بموضع. في كل موضع نضرب النافذة في الـ Kernel عنصراً بعنصر ونجمع كل الحاصلات → رقم واحد في خريطة الميزات. الأبعاد تتقلّص لأن النافذة لا تستطيع تجاوز حدود الصورة، ومن هنا جاءت معادلة `H - K + 1`.

</div>

#### Reference Implementation

```python
def my_conv2d(image, kernel):
    H, W = image.shape
    K = kernel.shape[0]
    Hp, Wp = H - K + 1, W - K + 1
    out = torch.zeros(Hp, Wp)
    for i in range(Hp):
        for j in range(Wp):
            region = image[i:i+K, j:j+K]      # 1) الـ Kernel يجلس فوق منطقة
            out[i, j] = (region * kernel).sum()  # 2) نضرب  3) نجمع
    return out

def conv_output_shape(H, W, K):
    return (H - K + 1, W - K + 1)

# --- اختبار ومطابقة مع PyTorch ---
test_image  = torch.arange(36, dtype=torch.float32).reshape(6, 6)
test_kernel = torch.tensor([[1., 0., -1.], [1., 0., -1.], [1., 0., -1.]])

mine = my_conv2d(test_image, test_kernel)
ref  = F.conv2d(test_image.view(1, 1, 6, 6), test_kernel.view(1, 1, 3, 3)).view(4, 4)
print("الشكل:", tuple(mine.shape), "| مطابق لـ F.conv2d:", torch.allclose(mine, ref))

# --- تطبيق ثلاث Kernels على رقم من MNIST ---
digit, digit_label = test_dataset[0]
digit = digit[0]                                   # (28, 28)

vertical_kernel   = torch.tensor([[1., 0., -1.], [1., 0., -1.], [1., 0., -1.]])
horizontal_kernel = vertical_kernel.t()
blur_kernel       = torch.ones(3, 3) / 9

feature_maps = torch.stack([
    my_conv2d(digit, vertical_kernel),
    my_conv2d(digit, horizontal_kernel),
    my_conv2d(digit, blur_kernel),
])
print("feature_maps.shape =", tuple(feature_maps.shape))

# --- العرض ---
titles = [ar('الأصلية'), ar('حواف رأسية'), ar('حواف أفقية'), ar('تنعيم')]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(digit, cmap='gray')
for idx in range(3):
    axes[idx + 1].imshow(feature_maps[idx], cmap='viridis')
for axi, t in zip(axes, titles):
    axi.set_title(t); axi.axis('off')
plt.suptitle(ar(f'الرقم {digit_label}: نفس الصورة، ثلاث Kernels، ثلاث خرائط ميزات'), fontsize=14)
plt.tight_layout(); plt.show()
```

#### Explanation

<div dir="rtl">

- **الـ Kernel الرأسية** `[1,0,-1]` تطرح العمود الأيمن من الأيسر: إن كانت المنطقة متجانسة فالناتج ≈ 0، وعند وجود حافة رأسية يصبح الناتج كبيراً.
- **المنقول (`.t()`)** يحوّل نفس المنطق إلى الاتجاه الأفقي.
- **kernel التنعيم** مجموعها 1 فتُعطي متوسط النافذة — نسخة ضبابية.
- الأبعاد: `28 - 3 + 1 = 26`. وثلاث Kernels ⇒ ثلاث قنوات خرج، تماماً كما في المعادلة `C' = عدد الـ Kernels`.
- لاحظ أن الـ Kernel **نفسها** تُطبَّق على كل المواضع — هذه هي **Parameter Sharing**، وأن كل خرج يعتمد على 9 بكسلات فقط — هذه هي **Local Connectivity**.

</div>

#### Expected Result

```
الشكل: (4, 4) | مطابق لـ F.conv2d: True
feature_maps.shape = (3, 26, 26)
```

<div dir="rtl">

مع رسم يُظهر أربع لوحات: الرقم الأصلي وثلاث خرائط ميزات مختلفة تماماً في مظهرها.

</div>

#### Common Mistakes

<div dir="rtl">

- استخدام `@` أو `torch.matmul` بدل الضرب عنصراً بعنصر `*`.
- كتابة `range(H)` بدل `range(H - K + 1)` فتخرج النافذة عن حدود الصورة وتُنتج نتائج مبتورة.
- نسيان `.sum()` فيُخزَّن tensor بأكمله في خانة واحدة (خطأ في الأبعاد).
- تمرير الصورة بشكل `(1, 28, 28)` بدل `(28, 28)` إلى `my_conv2d`.

</div>

</details>

---

<div dir="rtl">
<h2>Exercise 10 — التجميع الأعظمي وبناء شبكة CNN كاملة | Max Pooling &amp; The Full CNN</h2>

<h3 style="direction: rtl;">🎯 Learning Objective</h3>
<p style="direction: rtl; text-align: right;">أن تنفّذ <strong>Max Pooling</strong> بنفسك، ثم تبني <strong>شبكة CNN كاملة</strong> وتدرّبها على MNIST، وتقارنها وجهاً لوجه بالشبكة العادية في <strong>الدقة</strong> و <strong>عدد الأوزان</strong>، وأخيراً تفتح الشبكة وتنظر إلى ما تعلّمته الكيرنلات فعلاً.</p>

<h3 style="direction: rtl;">🧩 Concept</h3>
<p style="direction: rtl; text-align: right;"><strong>التجميع الأعظمي (Max Pooling)</strong>، <strong>بنية CNN (Conv → ReLU → Pool)</strong>، <strong>تراكم الطبقات</strong>, <strong>CNN مقابل Fully Connected</strong>، و <strong>Inductive Biases</strong>: Parameter Sharing و Translation Invariance و Hierarchical Features.</p>

<div align="center">
<img src="media/06-maxpooling.gif" width="560">
</div>

<div align="center">
<img src="media/04-full-cnn-pipeline.png" width="680">
</div>

<h3 style="direction: rtl;">📖 Scenario</h3>
<p style="direction: rtl; text-align: right;">كل ما بنيناه حتى الآن يجتمع في هذا التمرين: التمرير الأمامي والخلفي، الخسارة، المُحسّن، حلقة التدريب، الالتفاف، والتجميع. سنبني الشبكة الالتفافية وندرّبها على <strong>نفس بيانات</strong> الشبكة العادية، بـ <strong>دورات أقل</strong>، وسنرى هل تصمد الادعاءات النظرية أمام التجربة.</p>

<h3 style="direction: rtl;">📝 Task</h3>
<ol style="direction: rtl; text-align: right;">
<li>اكتب دالة <code>my_max_pool2d(fmap)</code> تُطبّق نافذة <code>2×2</code> بخطوة (stride) = 2 <strong>بدون تداخل</strong>:
   <ul style="direction: rtl; text-align: right; margin-top: 5px;">
   <li>المخرج بحجم <code>(H//2, W//2)</code>، وقيمة كل خانة = <strong>أعظم</strong> قيمة في النافذة المقابلة.</li>
   </ul>
</li>
<li>اختبرها على خريطة الميزات <code>feature_maps[0]</code> من التمرين السابق (اقتطعها إلى <code>26×26</code> → المخرج <code>13×13</code>)، وتحقّق من المطابقة مع <code>F.max_pool2d</code>.</li>
<li>ابنِ <code>cnn_model</code> بـ <code>nn.Sequential</code> بالبنية التالية بالضبط:</li>
</ol>

<table style="direction: rtl; text-align: right; width: 100%; border-collapse: collapse; margin: 10px 0;">
<tr style="background-color: #f0f0f0;">
<th style="border: 1px solid #ddd; padding: 8px; text-align: center;">الطبقة</th>
<th style="border: 1px solid #ddd; padding: 8px; text-align: center;">التحويل</th>
</tr>
<tr>
<td style="border: 1px solid #ddd; padding: 8px;"><code>Conv2d(1, 16, kernel_size=3, padding=1)</code> + <code>ReLU</code></td>
<td style="border: 1px solid #ddd; padding: 8px; direction: ltr; text-align: left;">28×28×1 → 28×28×16</td>
</tr>
<tr>
<td style="border: 1px solid #ddd; padding: 8px;"><code>MaxPool2d(2)</code></td>
<td style="border: 1px solid #ddd; padding: 8px; direction: ltr; text-align: left;">→ 14×14×16</td>
</tr>
<tr>
<td style="border: 1px solid #ddd; padding: 8px;"><code>Conv2d(16, 32, kernel_size=3, padding=1)</code> + <code>ReLU</code></td>
<td style="border: 1px solid #ddd; padding: 8px; direction: ltr; text-align: left;">→ 14×14×32</td>
</tr>
<tr>
<td style="border: 1px solid #ddd; padding: 8px;"><code>MaxPool2d(2)</code></td>
<td style="border: 1px solid #ddd; padding: 8px; direction: ltr; text-align: left;">→ 7×7×32</td>
</tr>
<tr>
<td style="border: 1px solid #ddd; padding: 8px;"><code>Flatten</code></td>
<td style="border: 1px solid #ddd; padding: 8px; direction: ltr; text-align: left;">→ 1568</td>
</tr>
<tr>
<td style="border: 1px solid #ddd; padding: 8px;"><code>Linear(32*7*7, 10)</code></td>
<td style="border: 1px solid #ddd; padding: 8px; direction: ltr; text-align: left;">→ 10</td>
</tr>
</table>

<ol style="direction: rtl; text-align: right;" start="4">
<li>احسب <code>cnn_params</code> وقارنه بـ <code>fc_params</code> من التمرين السادس.</li>
<li>درّب الشبكة <strong>دورتين فقط (epochs = 2)</strong> بنفس حلقة التدريب ذات الخطوات الأربع (<code>CrossEntropyLoss</code> + <code>Adam(lr=0.001)</code>) — <strong>لا حاجة لتسطيح الصورة قبل إدخالها</strong>.</li>
<li>قيّمها على <code>test_loader</code> واحسب <code>cnn_accuracy</code>، واطبع جدول المواجهة النهائي: الدقة وعدد الأوزان للشبكتين.</li>
<li><strong>انظر داخل الشبكة:</strong> مرّر صورة واحدة عبر <code>cnn_model[0]</code> مع <code>torch.relu</code>، وخزّن الناتج في <code>first_layer_maps</code> بشكل <code>(16, 28, 28)</code>، واعرض الـ 16 خريطة ميزات في شبكة <code>4×4</code>.</li>
</ol>

<h3 style="direction: rtl;">Requirements</h3>
<ul style="direction: rtl; text-align: right;">
<li><strong>المكتبات:</strong> <code>torch</code>, <code>torch.nn as nn</code>, <code>torch.nn.functional as F</code>, <code>matplotlib.pyplot</code>, <code>tqdm</code>.</li>
<li><strong>الدالة المطلوبة:</strong> <code>my_max_pool2d(fmap)</code>.</li>
<li><strong>المتغيرات المطلوبة:</strong> <code>cnn_model</code>, <code>cnn_params</code>, <code>cnn_accuracy</code>, <code>first_layer_maps</code>.</li>
<li><strong>الأبعاد:</strong> خرج <code>cnn_model</code> على دفعة = <code>(64, 10)</code>، و <code>first_layer_maps.shape == (16, 28, 28)</code>.</li>
<li><strong>قيود:</strong> استخدم نفس <code>train_loader</code> و <code>test_loader</code> ونفس المُحسّن ومعدل التعلم لتكون المقارنة عادلة. ممنوع استخدام <code>F.max_pool2d</code> داخل <code>my_max_pool2d</code> (تُستخدم للتحقق فقط).</li>
</ul>

<h3 style="direction: rtl;">Expected Result</h3>
<ul style="direction: rtl; text-align: right;">
<li>النتيجة المتوقعة هي النقطة التي يجتمع فيها الدرس كله:</li>
<ul style="direction: rtl; text-align: right; list-style-type: disc;">
<li>شبكة CNN تحقق دقة <strong>أعلى</strong> من الشبكة العادية،</li>
<li>رغم أنها تحتوي على عدد أوزان <strong>أقل بعدة أضعاف</strong> (بفضل Parameter Sharing)،</li>
<li>ورغم أنها تدرّبت <strong>دورات أقل</strong>.</li>
</ul>
<li>وفي عرض خرائط الميزات الستة عشر سترى أن <strong>كل كيرنل ترى الرقم بطريقة مختلفة</strong>: بعضها يُضيء عند الحواف الرأسية، وبعضها عند الأفقية، وبعضها يبرز الخلفية — وكل هذا <strong>تعلّمته الشبكة بنفسها</strong> ولم نُخبرها به.</li>
</ul>

</div>

### 💡 Hints

<details>
<summary>💡 Hint 1</summary>

<div dir="rtl">

في Max Pooling، النافذة تقفز بمقدار 2 وليس 1. لذا حلقتك تمر على `range(H // 2)` والنافذة تبدأ عند `2*i`:
`fmap[2*i : 2*i+2, 2*j : 2*j+2].max()`.

</div>

</details>

<details>
<summary>💡 Hint 2</summary>

<div dir="rtl">

لماذا `Linear(32 * 7 * 7, 10)`؟ لأن `padding=1` يحافظ على الأبعاد بعد كل Conv (28 تبقى 28)، بينما كل `MaxPool2d(2)` يقسمها على 2: `28 → 14 → 7`. وعدد القنوات بعد الطبقة الثانية = 32. إذن `Flatten` يُنتج `32 × 7 × 7 = 1568`.

</div>

</details>

<details>
<summary>💡 Hint 3</summary>

<div dir="rtl">

لاستخراج خرائط الطبقة الأولى:

```python
sample_img, sample_label = test_dataset[0]
with torch.no_grad():
    first_layer_maps = torch.relu(cnn_model[0](sample_img.unsqueeze(0)))[0]  # (16, 28, 28)
```

`unsqueeze(0)` تُضيف بُعد الدفعة، و `[0]` في النهاية تزيله من الناتج.

</div>

</details>

### 🧑‍💻 Your Solution

In [ ]:
# TODO: أكمل التنفيذ

# Step 1: Max Pooling من الصفر (نافذة 2×2، stride = 2)
def my_max_pool2d(fmap):
    pass

# Step 2: اختبرها وقارنها مع F.max_pool2d

# Step 3: ابنِ شبكة CNN
cnn_model = None

# Step 4: احسب عدد الأوزان وقارنه بالشبكة العادية
cnn_params = None

# Step 5: درّبها دورتين بنفس حلقة التدريب ذات الخطوات الأربع

# Step 6: قيّمها واطبع جدول المواجهة
cnn_accuracy = None

# Step 7: انظر داخل الشبكة — 16 خريطة ميزات من الطبقة الأولى
first_layer_maps = None

# Write your solution below

### ✅ Self-Check

In [ ]:
# --- Max Pooling ---
assert 'my_max_pool2d' in dir(), "الدالة 'my_max_pool2d' لم تُعرَّف."
_test_map = feature_maps[0][:26, :26]
_pooled   = my_max_pool2d(_test_map)
_ref_pool = F.max_pool2d(_test_map.view(1, 1, 26, 26), 2).view(13, 13)
assert tuple(_pooled.shape) == (13, 13), \
    f"شكل مخرج التجميع خاطئ: {tuple(_pooled.shape)} (المتوقع (13, 13) — النصف)."
assert torch.allclose(_pooled, _ref_pool, atol=1e-5), \
    "نتيجتك لا تطابق F.max_pool2d — تأكد من stride=2 (نوافذ بلا تداخل) ومن أخذ max وليس mean."

# --- بنية الشبكة ---
assert cnn_model is not None, "الشبكة 'cnn_model' لم تُبنَ."
assert isinstance(cnn_model[0], nn.Conv2d), "الطبقة الأولى يجب أن تكون nn.Conv2d وليست Flatten."
assert cnn_model[0].in_channels == 1 and cnn_model[0].out_channels == 16, \
    "الطبقة الأولى: قناة مدخل واحدة (grayscale) و 16 كيرنل."

_out = cnn_model(images)
assert tuple(_out.shape) == (64, 10), f"خرج الشبكة خاطئ: {tuple(_out.shape)} (المتوقع (64, 10))."

assert cnn_params is not None, "المتغير 'cnn_params' لم يُحسب."
assert cnn_params < fc_params, (
    f"عدد أوزان CNN ({cnn_params:,}) يجب أن يكون أقل من الشبكة العادية ({fc_params:,}) "
    "بفضل Parameter Sharing.")

# --- الأداء ---
assert cnn_accuracy is not None, "المتغير 'cnn_accuracy' لم يُحسب."
assert 0.0 <= cnn_accuracy <= 1.0, f"الدقة يجب أن تكون نسبة بين 0 و 1، لكنها {cnn_accuracy}."
assert cnn_accuracy > 0.97, f"دقة CNN منخفضة: {cnn_accuracy:.2%} (المتوقع > 97%)."
assert cnn_accuracy >= fc_accuracy, (
    f"CNN ({cnn_accuracy:.2%}) لم تتفوق على الشبكة العادية ({fc_accuracy:.2%}) — "
    "تحقق من البنية وحلقة التدريب.")

# --- خرائط الميزات ---
assert first_layer_maps is not None and tuple(first_layer_maps.shape) == (16, 28, 28), (
    f"شكل first_layer_maps خاطئ: "
    f"{tuple(first_layer_maps.shape) if first_layer_maps is not None else None} (المتوقع (16, 28, 28)).")
assert (first_layer_maps >= 0).all(), "بعد ReLU يجب ألا تحتوي الخرائط على قيم سالبة."

print("✅ Basic checks passed.")
print(f"   Fully Connected | دقة {fc_accuracy:.2%} | {fc_params:>8,} وزن")
print(f"   CNN             | دقة {cnn_accuracy:.2%} | {cnn_params:>8,} وزن")
print(f"   👉 CNN أصغر بـ {fc_params/cnn_params:.1f} مرة وأدق!")

### 💡 Solution

<details>
<summary>💡 Show Solution</summary>

#### Approach

<div dir="rtl">

نُنفّذ Max Pooling بنوافذ غير متداخلة، ثم نبني شبكة تُكرّر الثنائي **Conv → ReLU → Pool** مرتين: الأبعاد المكانية تصغر (28→14→7) بينما عدد القنوات يزيد (1→16→32). التسطيح يأتي **في النهاية فقط** — بعد أن استخرجنا الميزات المكانية، لا قبلها.

</div>

#### Reference Implementation

```python
# ===== Step 1: Max Pooling من الصفر =====
def my_max_pool2d(fmap):
    H, W = fmap.shape
    out = torch.zeros(H // 2, W // 2)
    for i in range(H // 2):
        for j in range(W // 2):
            window = fmap[2*i : 2*i+2, 2*j : 2*j+2]   # نافذة 2×2 بلا تداخل
            out[i, j] = window.max()                  # أقوى استجابة فقط
    return out

test_map = feature_maps[0][:26, :26]
pooled   = my_max_pool2d(test_map)
ref_pool = F.max_pool2d(test_map.view(1, 1, 26, 26), 2).view(13, 13)
print(f"{tuple(test_map.shape)} → {tuple(pooled.shape)} | مطابق: {torch.allclose(pooled, ref_pool)}")

# ===== Step 3: بناء شبكة CNN =====
torch.manual_seed(42)
cnn_model = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1),   # 28×28×1  → 28×28×16
    nn.ReLU(),
    nn.MaxPool2d(2),                              # → 14×14×16
    nn.Conv2d(16, 32, kernel_size=3, padding=1),  # → 14×14×32
    nn.ReLU(),
    nn.MaxPool2d(2),                              # → 7×7×32
    nn.Flatten(),                                 # → 1568 (الآن فقط نسطّح!)
    nn.Linear(32 * 7 * 7, 10)                     # → 10 أرقام
)
print(cnn_model)

cnn_params = sum(p.numel() for p in cnn_model.parameters())
print(f"\n👥 أوزان CNN: {cnn_params:,} | أوزان الشبكة العادية: {fc_params:,}")
print(f"💡 CNN أصغر بـ {fc_params/cnn_params:.1f} مرة — بفضل Parameter Sharing!")

# ===== Step 5: التدريب — نفس الخطوات الأربع =====
loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cnn_model.parameters(), lr=0.001)
epochs = 2

for epoch in range(epochs):
    total_correct, total_samples = 0, 0
    progress = tqdm(train_loader, desc=f'الدورة {epoch+1}/{epochs}')
    for imgs, lbls in progress:
        optimizer.zero_grad()            # 1
        outputs = cnn_model(imgs)        # 2 — لا حاجة لتسطيح الصورة!
        loss = loss_fn(outputs, lbls)
        loss.backward()                  # 3
        optimizer.step()                 # 4
        _, predicted = torch.max(outputs, 1)
        total_correct += (predicted == lbls).sum().item()
        total_samples += lbls.size(0)
        progress.set_postfix({'الدقة': f'{total_correct/total_samples:.3f}'})
    print(f"الدورة {epoch+1}: دقة {total_correct/total_samples:.2%}")

# ===== Step 6: المواجهة =====
cnn_model.eval()
correct, total = 0, 0
with torch.no_grad():
    for imgs, lbls in tqdm(test_loader, desc='اختبار CNN'):
        _, predicted = torch.max(cnn_model(imgs), 1)
        total   += lbls.size(0)
        correct += (predicted == lbls).sum().item()
cnn_accuracy = correct / total

print("\n" + "=" * 52)
print("           🥊 النتيجة النهائية للمواجهة 🥊")
print("=" * 52)
print(f"  الشبكة العادية (FC) | دقة {fc_accuracy:.2%} | {fc_params:>8,} وزن")
print(f"  الشبكة الالتفافية   | دقة {cnn_accuracy:.2%} | {cnn_params:>8,} وزن")
print("=" * 52)

# ===== Step 7: ماذا تعلّمت الكيرنلات؟ =====
sample_img, sample_label = test_dataset[0]
with torch.no_grad():
    first_layer_maps = torch.relu(cnn_model[0](sample_img.unsqueeze(0)))[0]   # (16, 28, 28)

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for k, axi in enumerate(axes.flat):
    axi.imshow(first_layer_maps[k], cmap='viridis')
    axi.set_title(f'Map {k+1}', fontsize=9)
    axi.axis('off')
plt.suptitle(ar(f'الـ 16 خريطة ميزات للطبقة الأولى — المدخل: الرقم {sample_label}'), fontsize=14)
plt.tight_layout(); plt.show()

cnn_model.train()
```

#### Explanation

<div dir="rtl">

- **Max Pooling** يُبقي "هل وُجد النمط؟" ويتخلّى عن "أين بالضبط؟" — وهذا مصدر **Translation Invariance**.
- **`padding=1`** يضيف إطاراً من الأصفار فيحافظ على الأبعاد بعد الالتفاف (28 تبقى 28)، فنتحكّم نحن بموعد التصغير عبر الـ Pooling.
- **عدد الأوزان:** كل كيرنل في الطبقة الأولى فيها `3×3 = 9` أوزان فقط، وتُطبَّق على كل مواضع الصورة — بينما طبقة `Linear(784, 128)` وحدها فيها أكثر من مئة ألف وزن.
- **خرائط الميزات** تُثبت أن كل كيرنل تعلّمت كاشفاً مختلفاً بنفسها: حواف بزوايا مختلفة، مناطق ساطعة، خلفية. هذا هو **التسلسل الهرمي للميزات** في طبقته الأولى.

</div>

#### Expected Result

```
(26, 26) → (13, 13) | مطابق: True
👥 أوزان CNN: 20,490 | أوزان الشبكة العادية: 109,386
💡 CNN أصغر بـ 5.3 مرة — بفضل Parameter Sharing!
====================================================
           🥊 النتيجة النهائية للمواجهة 🥊
====================================================
  الشبكة العادية (FC) | دقة ~97% |  109,386 وزن
  الشبكة الالتفافية   | دقة ~98-99% |   20,490 وزن
====================================================
```

#### Common Mistakes

<div dir="rtl">

- استخدام `stride=1` في التجميع (نوافذ متداخلة) فلا تتقلّص الأبعاد إلى النصف.
- كتابة `nn.Linear(32*14*14, 10)` أو `nn.Linear(784, 10)` بدل `nn.Linear(32*7*7, 10)` — احسب الأبعاد طبقةً بطبقة قبل كتابة الرقم.
- وضع `nn.Flatten()` في **بداية** الشبكة (عادة قديمة من التمرين السادس!) — عندها تفقد كل فائدة الالتفاف.
- نسيان `padding=1` فتصبح الأبعاد `26 → 13 → 11 → 5` ويفشل `Linear` بخطأ في الأبعاد.
- مقارنة CNN مُدرَّبة 10 دورات بشبكة عادية مُدرَّبة دورة واحدة — المقارنة يجب أن تكون بنفس الشروط أو في صالح الشبكة العادية.

</div>

</details>

---

<div dir="rtl">
<h2>Skills Checklist</h2>

<p style="direction: rtl; text-align: right;">ضع علامة أمام كل مهارة أتقنتها فعلياً بالكود:</p>

<ul style="direction: rtl; text-align: right; list-style-type: none; padding-right: 0;">
<li style="margin-bottom: 5px;">☐ <strong>Computational Graph</strong> — تفكيك معادلة إلى بوابات (Exercise 1)</li>
<li style="margin-bottom: 5px;">☐ <strong>Forward Pass</strong> — التمرير الأمامي وحفظ القيم الوسيطة (Exercise 1)</li>
<li style="margin-bottom: 5px;">☐ <strong>Derivatives as Sensitivity</strong> — المشتقة العددية (Exercise 1)</li>
<li style="margin-bottom: 5px;">☐ <strong>Backward Pass</strong> — التمرير الخلفي عبر البوابات (Exercise 2)</li>
<li style="margin-bottom: 5px;">☐ <strong>Chain Rule</strong> — قاعدة السلسلة (Exercise 2)</li>
<li style="margin-bottom: 5px;">☐ <strong>Gate Behaviors</strong> — الجمع يوزّع، الضرب يبدّل، الأعظم يوجّه (Exercise 2)</li>
<li style="margin-bottom: 5px;">☐ <strong>Gradient Accumulation at Branches</strong> — تجميع التدرجات عند التفرّع (Exercise 3)</li>
<li style="margin-bottom: 5px;">☐ <strong>Autograd</strong> — <code>requires_grad</code> و <code>.backward()</code> و <code>.grad</code> (Exercise 3)</li>
<li style="margin-bottom: 5px;">☐ <strong>MSE Loss</strong> — متوسط مربعات الخطأ (Exercise 4)</li>
<li style="margin-bottom: 5px;">☐ <strong>Cross-Entropy Loss &amp; Softmax</strong> — الإنتروبيا المتقاطعة (Exercise 4)</li>
<li style="margin-bottom: 5px;">☐ <strong>Gradient Descent Update Rule</strong> — قاعدة تحديث الأوزان (Exercise 5)</li>
<li style="margin-bottom: 5px;">☐ <strong>Learning Rate</strong> — أثر حجم الخطوة والتباعد (Exercise 5)</li>
<li style="margin-bottom: 5px;">☐ <strong>MNIST Data Pipeline</strong> — Transforms و Normalize و DataLoader (Exercise 6)</li>
<li style="margin-bottom: 5px;">☐ <strong>Fully Connected Network</strong> — بناء الشبكة وحساب المعاملات (Exercise 6)</li>
<li style="margin-bottom: 5px;">☐ <strong>Training Loop</strong> — الخطوات الأربع (Exercise 7)</li>
<li style="margin-bottom: 5px;">☐ <strong>Model Evaluation</strong> — <code>torch.no_grad()</code> والدقة على بيانات جديدة (Exercise 7)</li>
<li style="margin-bottom: 5px;">☐ <strong>The Flattening Problem</strong> — إثبات ضياع البنية المكانية (Exercise 8)</li>
<li style="margin-bottom: 5px;">☐ <strong>2D Convolution</strong> — الالتفاف من الصفر (Exercise 9)</li>
<li style="margin-bottom: 5px;">☐ <strong>Kernels &amp; Feature Maps</strong> — Kernels متعددة وقنوات الخرج (Exercise 9)</li>
<li style="margin-bottom: 5px;">☐ <strong>Output Shape Formula</strong> — $H' = H - K + 1$ (Exercise 9)</li>
<li style="margin-bottom: 5px;">☐ <strong>Max Pooling</strong> — التجميع الأعظمي من الصفر (Exercise 10)</li>
<li style="margin-bottom: 5px;">☐ <strong>CNN Architecture</strong> — Conv → ReLU → Pool → Flatten → Linear (Exercise 10)</li>
<li style="margin-bottom: 5px;">☐ <strong>Parameter Sharing &amp; Local Connectivity</strong> — لماذا CNN أصغر (Exercise 10)</li>
<li style="margin-bottom: 5px;">☐ <strong>CNN vs Fully Connected</strong> — المقارنة العملية (Exercise 10)</li>
</ul>

</div>

---

<div dir="rtl">
<h2>Completion Summary</h2>

<p style="direction: rtl; text-align: right;">بإتمامك التمارين العشرة تكون قد قطعت الطريق كاملاً من <strong>الرياضيات إلى النموذج العامل</strong>:</p>

<p style="direction: rtl; text-align: right;">بدأت بـ <strong>دارة من أربع بوابات</strong> حسبت فيها التمرير الأمامي والخلفي بيدك، وفهمت أن <code>loss.backward()</code> ليست سحراً بل ضرب أرقام بقاعدة السلسلة. ثم اكتشفت أن التدرجات <strong>تُجمع عند التفرّع</strong> — وهو السبب المباشر لوجود <code>optimizer.zero_grad()</code> في كل حلقة تدريب في العالم.</p>

<p style="direction: rtl; text-align: right;">بعدها بنيت <strong>دوال الخسارة</strong> بنفسك وأثبتّ أنها تطابق دوال PyTorch، ونفّذت <strong>قاعدة تحديث المُحسّن</strong> ورأيت بأم عينك كيف ينفجر التدريب مع معدل تعلم كبير.</p>

<p style="direction: rtl; text-align: right;">ثم انتقلت إلى التطبيق: جهّزت <strong>MNIST</strong>، بنيت شبكة <strong>Fully Connected</strong>، وكتبت <strong>حلقة التدريب</strong> بخطواتها الأربع، وحققت دقة تتجاوز 95%. لكنك لم تتوقف عند النتيجة — بل <strong>اختبرت افتراضاتها</strong>: خلطت البكسلات وأثبتّ تجريبياً أن الشبكة لم تكن ترى الصورة أصلاً، بل قائمة أرقام.</p>

<p style="direction: rtl; text-align: right;">وهنا جاء الحل: نفّذت <strong>الالتفاف</strong> و <strong>التجميع الأعظمي</strong> من الصفر، وبنيت <strong>شبكة CNN</strong> حققت دقة أعلى بأوزان أقل ودورات أقل — ثم فتحت الطبقة الأولى ونظرت إلى الكيرنلات الستة عشر التي <strong>تعلّمت كواشف الحواف بنفسها</strong> دون أن يُخبرها أحد.</p>

<p style="direction: rtl; text-align: right;">هذه ليست تمارين منفصلة — إنها القصة الكاملة لكيفية تعلّم الآلة أن ترى. 🎓</p>

<p style="direction: rtl; text-align: right;"><strong>الخطوة التالية:</strong> جرّب التحديات المفتوحة في نهاية نوتبوك الأسبوع الخامس — غيّر عدد الكيرنلات، احذف طبقات الـ Pooling واحسب الأبعاد الجديدة على الورق أولاً، أو جرّب <strong>Fashion-MNIST</strong> بتغيير سطر واحد في تحميل البيانات وقارن أي الشبكتين تتأثر أكثر بصعوبة البيانات.</p>

<p style="direction: rtl; text-align: center; font-style: italic; margin-top: 20px;"><em>Computer Vision Course — Week 5 | من الانتشار الخلفي إلى الشبكات الالتفافية</em></p>

</div>

---